## Retail Chain Profit Leakage & Demand Optimization
### Stage 1: Data Cleaning

This notebook contains the cleaning pipeline for the retail chain
profit leakage and demand optimization project.

The raw Excel dataset is loaded without modification, and separate
copies of each table are created for cleaning. The original raw data
is preserved throughout the process.

In [1]:
import pandas as pd
file_path = r"C:\Users\manit\OneDrive\Desktop\PROJECT_1\Retail_Profit_Leakage_Dataset.xlsx"
excel_data = pd.ExcelFile(file_path)

In [2]:
sales = pd.read_excel(excel_data, sheet_name="Sales_Transactions")
product = pd.read_excel(excel_data, sheet_name="Product_Master")
inventory = pd.read_excel(excel_data, sheet_name="Inventory_Daily")
returns = pd.read_excel(excel_data, sheet_name="Returns")
vendor = pd.read_excel(excel_data, sheet_name="Vendor_Supply")
promotions = pd.read_excel(excel_data, sheet_name="Promotions")
customer = pd.read_excel(excel_data, sheet_name="Customer_Master")

In [3]:
# create the clean copies
sales_clean = sales.copy()
product_clean = product.copy()
inventory_clean = inventory.copy()
returns_clean = returns.copy()
vendor_clean = vendor.copy()
promotions_clean = promotions.copy()
customer_clean = customer.copy()

In [4]:
# sales data duplicates finding
sales_clean.duplicated().sum()

np.int64(1217)

In [5]:
# product_clean data duplicates finding
product_clean.duplicated().sum()

np.int64(16)

In [6]:
# Inventory data duplicates finding
inventory_clean.duplicated().sum()

np.int64(424)

In [7]:
# returns data duplicates finding
returns_clean.duplicated().sum()

np.int64(84)

In [8]:
#  vendor data duplicates finding
vendor_clean.duplicated().sum()

np.int64(365)

In [9]:
# promotions data duplicates finding
promotions_clean.duplicated().sum()

np.int64(8)

In [10]:
# Customer data duplicates finding
customer_clean.duplicated().sum()

np.int64(264)

###  Cleaning step1: Remove exact duplicates

In [11]:
# drop_duplicates() removes duplicate rows from a Pandas DataFrame and
# keeps the first occurrence by default.

sales_clean=sales_clean.drop_duplicates()

In [12]:
# drop duplicate records in product_clean
product_clean = product_clean.drop_duplicates()

In [13]:
# drop duplicate records in inventory_clean
inventory_clean = inventory_clean.drop_duplicates()

In [14]:
# drop duplicate records in returns_clean
returns_clean = returns_clean.drop_duplicates()

In [15]:
# drop duplicate records in vendor_clean
vendor_clean = vendor_clean.drop_duplicates()

In [16]:
# drop duplicate records in promotions_clean
promotions_clean = promotions_clean.drop_duplicates()

In [17]:
# drop duplicate records in customer_clean
customer_clean = customer_clean.drop_duplicates()

In [18]:
print("sales_clean with no.of duplicate records:", sales_clean.duplicated().sum())
print("product_clean with no.of duplicate records:", product_clean.duplicated().sum())
print("inventory_clean with no.of duplicate records:", inventory_clean.duplicated().sum())
print("returns_clean with no.of duplicate records:", returns_clean.duplicated().sum())
print("vendor_clean with no.of duplicate records:", vendor_clean.duplicated().sum())
print("promotions_clean with no.of duplicate records:", promotions_clean.duplicated().sum())
print("customer_clean with no.of duplicate records:", customer_clean.duplicated().sum())

sales_clean with no.of duplicate records: 0
product_clean with no.of duplicate records: 0
inventory_clean with no.of duplicate records: 0
returns_clean with no.of duplicate records: 0
vendor_clean with no.of duplicate records: 0
promotions_clean with no.of duplicate records: 0
customer_clean with no.of duplicate records: 0


#### Duplicate Cleaning Result

Exact duplicate records were successfully removed from all seven
clean tables.
The raw source data was not modified.
Post-cleaning validation confirmed that all seven clean tables
contain zero exact duplicate records.

### Cleaning Step 2: Missing SKU Mappings

From our profiling notebook, we found:

2,016 sales rows had an unmatched SKU_ID initially.

After temporary standardization of spaces, case, and hyphens:

-> 1,995 sales rows remained unmatched.

-> 1,560 unique SKU IDs remained unmatched.

#### The SKU_ID in Sales Transactions is used to connect sales records with the Product Master. Some SKU values have inconsistent formatting, so they cannot be matched directly.
#### Before standardizing the values, the original SKU is preserved in a separate column for auditability. Standardization will only be used to identify reliable matches with the Product Master.

In [19]:
# craete a new columns for audit
sales_clean["Original_SKU_ID"] = sales_clean["SKU_ID"].copy()

In [20]:
product

,SKU_ID,SKU_Name,Category,Sub_Category,Brand,Tax_Percent,Unit_Cost,MRP
0,SKU00001,Snackify Biscuits 500g,Snacks,Biscuits,Snackify,12,55.38,99.00
1,SKU00002,FreshCo Paneer 250g,Dairy,Paneer,FreshCo,5,43.66,59.06
2,SKU00003,DailyDairy Edible Oil 500g,Groceries,Edible Oil,DailyDairy,5,76.77,105.78
3,SKU00004,Snackify Butter 1kg,Dairy,Butter,Snackify,5,144.24,259.33
4,SKU00005,BabyBloom Detergent 750ml,Home Care,Detergent,BabyBloom,18,112.73,175.81
...,...,...,...,...,...,...,...,...
1213,SKU00859,Snackify Floor Cleaner 100g,Home Care,Floor Cleaner,Snackify,18,94.47,166.26
1214,SKU01137,DailyDairy Meat 500g,Frozen,Meat,DailyDairy,18,106.15,149.18
1215,SKU00895,AquaFizz Powder Combo,Baby Care,Powder,AquaFizz,5,44.13,77.21
1216,SKU00364,FrostBite Cheese 1kg,Dairy,Cheese,FrostBite,5,64.02,101.28


In [21]:
# we don't want to simply change every SKU in Sales to our own standardized format.

#Because Product_Master is our reference/master table.

#We want SalesData to use the actual SKU_ID that exists in Product_Master.

#So we're building a lookup table(i.e sku_mappig) to check Standardied sku_id vs original master sku_id:
product_sku_standard = (product_clean["SKU_ID"].astype(str).str.strip().str.upper())
sku_mapping = pd.DataFrame({"Standardized_SKU": product_sku_standard,
                            "Product_SKU_ID": product_clean["SKU_ID"]})

display(sku_mapping.head(5))

,Standardized_SKU,Product_SKU_ID
0,SKU00001,SKU00001
1,SKU00002,SKU00002
2,SKU00003,SKU00003
3,SKU00004,SKU00004
4,SKU00005,SKU00005


In [22]:
# So we can do upper(), strip() to standardized versions for comparison.
sales_sku_standard = (sales_clean["SKU_ID"].astype(str).str.strip().str.upper())

product_sku_standard = (product_clean["SKU_ID"].astype(str).str.strip().str.upper())
sales_sku_standard.head(5)

0    SKU00565
1    SKU01082
2    SKU00901
3    SKU00524
4    SKU00215
Name: SKU_ID, dtype: object

In [23]:
product_sku_standard.head(5)

0    SKU00001
1    SKU00002
2    SKU00003
3    SKU00004
4    SKU00005
Name: SKU_ID, dtype: object

In [24]:
matched_sku = sales_sku_standard.isin(product_sku_standard)

print("Total sales rows:", len(sales_clean))
print("Matched SKU rows:", matched_sku.sum())
print("Unmatched SKU rows:", (~matched_sku).sum())

Total sales rows: 80383
Matched SKU rows: 78372
Unmatched SKU rows: 2011


#### After deleting duplicate records we have 2011 unmatched SKU sales rows. We should not simply delete those 2011 rows.

#### Why?

#### Those sales records are still legitimate sales records; we just don't know which product they belong to. Deleting them could affect:

 -Sales totals
 
 -Profit calculations
 
 -Category analysis
 
 -SKU-level analysis
 
 -Stockout analysis

### inspect the unmatched SKUs

In [25]:
# Among the 2011 unmatched sales records, what actual UNIQUE SKU values are appearing?
unmatched_sku_values= sales_clean.loc[~matched_sku,"SKU_ID"]
print("unmatched_sku_values:", unmatched_sku_values.nunique())
display(unmatched_sku_values.value_counts())

unmatched_sku_values: 1580


SKU_ID
SKU00178     70
SKU00585     64
SKU00154     64
SKU00755     59
SKU01189     59
             ..
SKU-23867     1
SKU-40235     1
SKU-45423     1
SKU-22744     1
SKU-19228     1
Name: count, Length: 1580, dtype: int64

In [26]:
# Check Are these 1,580 unmatched SKU IDs actually absent from Product Master, or 
# are they just represented differently?
unmatched_sku_sample= unmatched_sku_values.drop_duplicates()
product_sku_set= set(product_clean["SKU_ID"].astype(str).str.strip().str.upper())
comparison=pd.DataFrame({"Unmatched SKU_ID": unmatched_sku_sample,
                     "Exists_in_Product_Master":unmatched_sku_sample.astype(str).str.strip().str.upper().isin(product_sku_set)})
display(comparison)

,Unmatched SKU_ID,Exists_in_Product_Master
47,SKU00178,False
93,SKU-39322,False
110,SKU-95439,False
120,SKU-54918,False
121,SKU-98184,False
...,...,...
81442,SKU-80134,False
81461,SKU-92028,False
81525,SKU-52946,False
81569,SKU-91368,False


In [27]:
print("Unmatched SKUs that exist in Product Master:",comparison["Exists_in_Product_Master"].sum())

print("Unmatched SKUs genuinely absent from Product Master:",(~comparison["Exists_in_Product_Master"]).sum())

Unmatched SKUs that exist in Product Master: 0
Unmatched SKUs genuinely absent from Product Master: 1580


#### Missing SKU Mapping Flag:
Sales records with SKU IDs that cannot be matched to the Product Master are retained. A flag is added to identify these records so that downstream analysis can distinguish transactions with valid product mappings from transactions with missing product mappings.

In [28]:
# Matched → SKU exists in Product Master
# Missing → SKU does not exist in Product Master 
sales_clean["SKU_Mapping_Status"] = "Matched"
sales_clean.loc[~matched_sku,"SKU_Mapping_Status"] = "Missing"
sales_clean["SKU_Mapping_Status"].value_counts()

SKU_Mapping_Status
Matched    78372
Missing     2011
Name: count, dtype: int64

#### Missing SKU Mappings Result: 
#### After standardizing SKU identifiers, 78,372 sales records were successfully matched to the Product Master, while 2011 records remained unmatched and flaged as missing records rather than newly creating our own SKU_IDs in product_master dataset

### Cleaning step 3: Negative Quantity.
#### During our profiling, we found:

 - 326 sales records have Quantity < 0.

In [29]:
# we are keeping only row with Quantity < 0 and
# making a separate DataFrame and assigning to a variable

negative_quantity_sales=sales_clean[sales_clean["Quantity"]<0].copy()
print("Negative quantity records:", len (negative_quantity_sales))
display(negative_quantity_sales.head(20))

Negative quantity records: 326


,Order_ID,Order_Date,Store_ID,Customer_ID,SKU_ID,Quantity,Selling_Price,Discount_Amount,Promo_ID,Delivery_Type,Payment_Mode,Order_Status,Original_SKU_ID,SKU_Mapping_Status
336,ORD1000336,2025-01-19 00:00:00,PUN-05,CUST004249,SKU00815,-2,107.56,20.68,NaN,Home Delivery,UPI,Delivered,SKU00815,Matched
348,ORD1000348,2025-03-30 00:00:00,KOL-02,CUST011722,SKU00228,-2,63.63,6.52,NaN,Home Delivery,UPI,Cancelled,SKU00228,Matched
622,ORD1000622,2025-02-18 00:00:00,MUM-02,CUST010532,SKU00619,-2,86.34,43.98,PR0078,Home Delivery,Wallet,Delivered,SKU00619,Matched
669,ORD1000669,2025-03-22 00:00:00,AHM-03,CUST013544,SKU01021,-2,145.43,106.06,PR0053,Home Delivery,Card,Delivered,SKU01021,Matched
748,ORD1000748,2025-06-16 00:00:00,PUN-02,CUST006538,SKU01190,-5,38.87,26.20,PR0034,Home Delivery,UPI,Delivered,SKU01190,Matched
1422,ORD1001422,2025-06-22 00:00:00,KOL-05,CUST012657,SKU00139,-2,102.73,28.74,NaN,Home Delivery,UPI,Delivered,SKU00139,Matched
2186,ORD1002186,2025-05-26 00:00:00,CHE-04,CUST014075,SKU00617,-2,92.38,33.60,NaN,Home Delivery,Card,Delivered,SKU00617,Matched
2605,ORD1002605,2025-01-28 00:00:00,JAI-02,CUST003149,SKU00567,-4,89.28,2.28,NaN,Store Pickup,UPI,Delivered,SKU00567,Matched
2823,ORD1002823,2025-02-06 00:00:00,DEL-03,CUST005935,SKU01165,-2,132.07,32.26,NaN,Home Delivery,NetBanking,Delivered,SKU01165,Matched
3084,ORD1003084,2025-04-08 00:00:00,del5,CUST009198,SKU00864,-2,139.38,13.42,NaN,Home Delivery,UPI,Delivered,SKU00864,Matched


#### Negative Quantity vs Returns check:
Negative quantities may represent transaction reversals or returned products. Since the dataset contains a separate Returns table, the negative-quantity sales records are compared with the Returns table before deciding how they should be treated. If those 326 Order_IDs are present in Returns we arrive to a decision 

In [30]:
# Check whether negative-quantity orders are also present in the Returns table.
negative_order_ids = negative_quantity_sales["Order_ID"]
negative_orders_in_returns= negative_order_ids.isin(returns_clean["Order_ID"])
print("Negative quantity orders found in Returns:",negative_orders_in_returns.sum())
print("Negative quantity orders not found in Returns:",(~negative_orders_in_returns).sum())

Negative quantity orders found in Returns: 15
Negative quantity orders not found in Returns: 311


- So we cannot simply say that "every"negative quantity represents a return.
 we definitely should not simply convert for example -2 to 2, because that would change the meaning of the transaction.

#### Negative Quantity Order_Status Check:
The negative-quantity records are further examined by order status to determine whether they are associated with cancelled or completed sales transactions. This helps identify whether the negative quantities may represent transaction corrections or invalid sales records.

In [31]:
# count each value how many times it occured
negative_quantity_sales["Order_Status"].value_counts()

Order_Status
Delivered    304
Cancelled     22
Name: count, dtype: int64

#### Order Status Comparison finding:

The every negative-quantity record is NOT a cancelled record only 22 of them are cancelled. So 304 records tells meaningfull story so we should Flag Negative Quantity records for further filtering during analysis

In [32]:
# so changing like -3 to 3 would distort profits/sales calculations.
# so we flag them rather than converting or deleting

sales_clean["Negative_Quantity_Flag"] = "No"

sales_clean.loc[sales_clean["Quantity"] < 0,"Negative_Quantity_Flag"] = "Yes"

# validate
sales_clean["Negative_Quantity_Flag"].value_counts()

Negative_Quantity_Flag
No     80057
Yes      326
Name: count, dtype: int64

#### Negative Quantity Result:
#### A total of 326 sales records contain negative quantities. These records were retained in the cleaned dataset and flagged using Negative_Quantity_Flag. Fifteen records were linked to corresponding return records, while 311 had no matching return record. Since the reason for the remaining negative quantities could not be established from the available data, their original values were preserved.

### Cleaning step 4: Promo ID Mismatch Investigation:
Sales transactions containing a non-null Promo_ID are compared with the Promotions table. Unmatched promotion IDs are investigated to determine which promotion references are not available in the master promotion data.

In [33]:
# look at the records where Promo_ID actually exists and
# find Promo_id from sales table that are not present in Promotion table
# and do count ach Promo ID how many times it repeated
# reset_index converts series into normal DataFrame

unmatched_promo_sales= sales_clean[sales_clean["Promo_ID"].notna()& 
    ~sales_clean["Promo_ID"].isin(promotions_clean["Promo_ID"])].copy()
print("Unmatched promo sales records:", len(unmatched_promo_sales))
display(unmatched_promo_sales["Promo_ID"].value_counts()
       .rename_axis("Promo_ID")
       .reset_index(name="Sales_recound_count"))

Unmatched promo sales records: 3145


,Promo_ID,Sales_recound_count
0,PROMO-9999,2447
1,PR0067,152
2,PR0142,145
3,PR0183,145
4,PR0130,130
5,PR0133,126


In [34]:
unmatched_promo_sales

,Order_ID,Order_Date,Store_ID,Customer_ID,SKU_ID,Quantity,Selling_Price,Discount_Amount,Promo_ID,Delivery_Type,Payment_Mode,Order_Status,Original_SKU_ID,SKU_Mapping_Status,Negative_Quantity_Flag
0,ORD1000000,2025-06-01 00:00:00,HYD-03,CUST007684,SKU00565,4,72.73,19.52,PROMO-9999,Home Delivery,UPI,Delivered,SKU00565,Matched,No
15,ORD1000015,2025-03-13 00:00:00,AHM-05,CUST002601,SKU01053,2,159.42,27.72,PR0133,Store Pickup,UPI,Delivered,SKU01053,Matched,No
74,ORD1000074,2025-03-13 00:00:00,MUM-05,CUST013245,SKU00737,2,42.04,21.88,PROMO-9999,Express,Cash,Delivered,SKU00737,Matched,No
82,ORD1000082,2025-04-06 00:00:00,MUM-01,CUST013684,SKU00083,1,30.29,1.01,PROMO-9999,Home Delivery,Cash,Delivered,SKU00083,Matched,No
89,ORD1000089,2025-02-03 00:00:00,PUN-01,CUST008167,SKU00878,3,216.30,149.55,PR0133,Express,Cash,Delivered,SKU00878,Matched,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81513,ORD1069128,2025-03-23 00:00:00,CHE-03,CUST014816,SKU00055,2,28.30,23.62,PROMO-9999,Home Delivery,UPI,Delivered,SKU00055,Matched,No
81533,ORD1053146,2025-05-15 00:00:00,CHE-02,CUST010017,SKU00058,2,347.57,14.12,PROMO-9999,Home Delivery,Cash,Delivered,SKU00058,Matched,No
81563,ORD1050773,2025-04-18 00:00:00,PUN-02,CUST009424,SKU00883,2,45.82,18.84,PR0130,Store Pickup,Wallet,Delivered,SKU00883,Matched,No
81569,ORD1043398,2025-04-23 00:00:00,AHM-04,CUST009937,SKU-91368,4,232.79,18.56,PR0067,Store Pickup,Card,Delivered,SKU-91368,Missing,No


In [35]:
# "Show me whether those specific six Promo IDs exist anywhere in the Promotions master."
unmatched_promo_ids= unmatched_promo_sales["Promo_ID"].unique()
invalid_promo_df = pd.DataFrame(unmatched_promo_ids, columns=["Promo_ID"])
invalid_promo_df

,Promo_ID
0,PROMO-9999
1,PR0133
2,PR0142
3,PR0183
4,PR0130
5,PR0067


In [36]:
promotion_check=promotions_clean[promotions_clean["Promo_ID"].isin(unmatched_promo_ids)]
display(promotion_check)

,Promo_ID,Promo_Type,Start_Date,End_Date,Target_Category,Promo_Budget


In [37]:

promotion_check1=invalid_promo_df[invalid_promo_df["Promo_ID"].isin(promotions_clean["Promo_ID"])]
display(promotion_check1)

,Promo_ID


#### Promo ID Cleaning Treatment:

The six unmatched promotion IDs do not exist in the Promotions master table. The original values are preserved in a separate column for audit purposes. The invalid promotion references will be treated as missing in the cleaned data rather than being assigned to an unverified promotion.

In [38]:
sales_clean["Original_Promo_ID"]=sales_clean["Promo_ID"].copy()

In [39]:
invalid_promo_ids = unmatched_promo_sales["Promo_ID"].unique()

sales_clean.loc[ sales_clean["Promo_ID"].isin(invalid_promo_ids),"Promo_ID"] = pd.NA

In [40]:
sales_clean["Promo_Mapping_Status"] = "Matched"

# Sales where no Promo_ID was originally provided
sales_clean.loc[sales_clean["Original_Promo_ID"].isna(),"Promo_Mapping_Status"] = "No Promotion"

# Sales where an invalid Promo_ID was originally provided
sales_clean.loc[sales_clean["Original_Promo_ID"].isin(invalid_promo_ids),"Promo_Mapping_Status"] = "Missing"

In [41]:
sales_clean.head(20)

,Order_ID,Order_Date,Store_ID,Customer_ID,SKU_ID,Quantity,Selling_Price,Discount_Amount,Promo_ID,Delivery_Type,Payment_Mode,Order_Status,Original_SKU_ID,SKU_Mapping_Status,Negative_Quantity_Flag,Original_Promo_ID,Promo_Mapping_Status
0,ORD1000000,2025-06-01 00:00:00,HYD-03,CUST007684,SKU00565,4,72.73,19.52,<NA>,Home Delivery,UPI,Delivered,SKU00565,Matched,No,PROMO-9999,Missing
1,ORD1000001,2025-03-14 00:00:00,HYD-03,CUST004865,SKU01082,6,273.62,173.52,NaN,Store Pickup,Cash,Delivered,SKU01082,Matched,No,NaN,No Promotion
2,ORD1000002,2025-06-05 00:00:00,BEN-05,CUST006245,SKU00901,2,58.95,4.42,NaN,Store Pickup,Card,Delivered,SKU00901,Matched,No,NaN,No Promotion
3,ORD1000003,2025-06-23 00:00:00,JAI-02,CUST009196,SKU00524,2,188.08,23.20,NaN,Store Pickup,UPI,Cancelled,SKU00524,Matched,No,NaN,No Promotion
4,ORD1000004,2025-03-01 00:00:00,PUN-01,CUST000981,SKU00215,2,91.13,16.68,NaN,Home Delivery,Cash,Delivered,SKU00215,Matched,No,NaN,No Promotion
5,ORD1000005,2025-05-02 00:00:00,AHM-02,CUST005962,SKU01171,2,121.66,19.08,NaN,Store Pickup,UPI,Delivered,SKU01171,Matched,No,NaN,No Promotion
6,ORD1000006,2025-05-26 00:00:00,CHE-01,CUST010497,SKU00267,3,179.08,50.64,NaN,Store Pickup,Wallet,Delivered,SKU00267,Matched,No,NaN,No Promotion
7,ORD1000007,2025-06-01 00:00:00,CHE-01,CUST004609,SKU00418,4,263.93,279.84,PR0065,Store Pickup,UPI,Delivered,SKU00418,Matched,No,PR0065,Matched
8,ORD1000008,2025-04-13 00:00:00,CHE-03,CUST005707,SKU00540,2,62.93,3.86,NaN,Home Delivery,UPI,Delivered,SKU00540,Matched,No,NaN,No Promotion
9,ORD1000009,2025-06-16 00:00:00,BEN-03,CUST001059,SKU00396,2,149.88,112.80,PR0127,Home Delivery,NetBanking,Delivered,SKU00396,Matched,No,PR0127,Matched


In [42]:
# validate promo_mapping_status unique value counts
sales_clean["Promo_Mapping_Status"].value_counts()

Promo_Mapping_Status
No Promotion    50685
Matched         26553
Missing          3145
Name: count, dtype: int64

### Promo_ID cleaning Result:

- #### After removing duplicate records, 3,145 sales transactions contained invalid promotion IDs that were not present in the Promotions master table. 
- #### These records were retained, but their invalid Promo_ID values were set to missing and flagged as Missing.
- #### Records with an originally blank Promo_ID were separately classified as No Promotion. This prevents invalid promotion references from being used in promotion performance and ROI analysis.

### Cleaning Step 5: Returns without Sales Mapping

- During data validation, 150 return records were found with Order_ID values that
do not exist in the Sales Transactions table.
- These records are retained because deleting them could understate return activity
and refund losses. Instead, a Sales_Mapping_Status column is created to identify
whether each return can be matched to a sales transaction.

In [43]:
# Create a mapping status column for return records.
# Matched = Order_ID exists in Sales Transactions
# Missing = Order_ID does not exist in Sales Transactions


returns_clean["Sales_Mapping_Status"] = "Matched"
returns_clean.loc[~returns_clean["Order_ID"].isin(sales_clean["Order_ID"]),"Sales_Mapping_Status"]="Missing"

In [44]:
# Validate the cleaning
returns_clean["Sales_Mapping_Status"].value_counts()

Sales_Mapping_Status
Matched    4467
Missing     150
Name: count, dtype: int64

### Returns without Sales Mapping Result

#### The Returns table was successfully checked against the Sales Transactions table. The 150 previously identified returns without a matching sales Order_ID have been retained and classified as "Missing".

#### No return records were deleted. This preserves the original return information while clearly identifying records that cannot currently be linked to Sales.

###  Cleaning step 6: Vendor DatesVendor Date Cleaning

- The Vendor Supply table contains three date fields: PO_Date,
Expected_Delivery_Date, and Actual_Delivery_Date.

- During data validation, invalid date values were identified in these
columns. The date fields are converted to proper datetime format during
the cleaning stage. Invalid date values are converted to missing values
rather than being assigned an incorrect date.

In [45]:
# Convert vendor date columns to proper datetime format.
# Invalid date values are converted to NaT (Not a Time).


for i in ["PO_Date","Expected_Delivery_Date","Actual_Delivery_Date"]:
    vendor_clean[i]=pd.to_datetime(vendor_clean[i],errors="coerce")

#### Vendor Delivery Delay

Delivery delay is calculated as the difference between the Actual Delivery
Date and Expected Delivery Date.

- Positive values indicate late delivery
-  zero indicates delivery on the expected date
-  negative values indicate delivery before the expected date.

In [46]:
# Calculate delivery delay in days.
# Positive value = late delivery
# Zero = delivered on expected date
# Negative value = delivered early
# .dt.days because we are telling the substraction value is not ordinary integer it is pandas timedelta 

vendor_clean["Delay_Days"]=(vendor_clean["Actual_Delivery_Date"]-vendor_clean["Expected_Delivery_Date"]).dt.days

In [47]:
# Create Vendor SLA status
# Vendor SLA Breach % = Actual > Expected

vendor_clean["SLA_Status"]="On Time"
vendor_clean.loc[vendor_clean["Delay_Days"]>0,"SLA_Status"]="Late"
vendor_clean.loc[vendor_clean["Delay_Days"]<0,"SLA_Status"]="Early"
vendor_clean.loc[vendor_clean["Delay_Days"].isna(),"SLA_Status"]="Missing"

# validate each SLA value how many times it occured
vendor_clean["SLA_Status"].value_counts()



SLA_Status
On Time    20638
Late        8912
Missing      455
Early         80
Name: count, dtype: int64

In [48]:
vendor_clean["Delay_Days"].isna().sum()

np.int64(455)

### Vendor Date Cleaning Result:

#### The Vendor Supply date fields were converted to proper datetime format, with invalid date values converted to missing values. Delivery delay was then calculated by comparing Actual_Delivery_Date with Expected_Delivery_Date.

#### The cleaned data contains 8,912 late deliveries, 20,638 on-time deliveries, 80 early deliveries, and 455 records where delivery delay could not be calculated because the required dates were missing or invalid.

#### A SLA_Status field was created to support later vendor performance and SLA breach analysis.

### Cleaning step 7: Inconsistent Store_ID formats

We'll standardize Store_ID by:

- Converting to text
- Removing leading/trailing spaces
- Converting to uppercase

#### Store ID Standardization

The Sales Transactions table contains inconsistent Store_ID formats,
where the same store can appear with differences in capitalization,
hyphens, underscores, or surrounding spaces.

To ensure consistent store-level analysis and auditing, "the original Store_ID is
preserved in a separate column" and the Store_ID field is standardized.

In [49]:
# Preserve the original Store_ID before making any changes.
# This allows the original value to be retained for audit purposes.

sales_clean["Original_Store_ID"] = sales_clean["Store_ID"].copy()

In [50]:
# Standardize Store_ID values:
# 1. Convert to string
# 2. Remove leading/trailing spaces
# 3. Convert to uppercase
# 4. Remove hyphens
# 5. Remove underscores

sales_clean["Store_ID"] = (sales_clean["Store_ID"].astype(str).str.strip().str.upper())

In [51]:
# validating the reult 
sales_clean["Store_ID"].nunique()

150

In [52]:
# Check original vs cleaned values
sales_id_check=sales_clean[["Original_Store_ID", "Store_ID"]].drop_duplicates()
display(sales_id_check.head(20))

,Original_Store_ID,Store_ID
0,HYD-03,HYD-03
2,BEN-05,BEN-05
3,JAI-02,JAI-02
4,PUN-01,PUN-01
5,AHM-02,AHM-02
6,CHE-01,CHE-01
8,CHE-03,CHE-03
9,BEN-03,BEN-03
10,HYD-05,HYD-05
11,MUM-04,MUM-04


### Inconsistent Store_ID formats Result

#### Store_ID values were standardized by removing differences in capitalization, spaces, hyphens, and underscores. The original Store_ID values were preserved in the Original_Store_ID column for audit purposes.

#### The number of unique Store_ID representations was reduced from 462 to 100 standardized identifiers in the Sales Transactions data.

### Cleaning step 8: Cancelled Orders Appearing in Sales

#### Cancelled Orders in Sales Transactions

The Sales Transactions table contains cancelled orders that are still
present in the sales dataset. During data validation, 5,761 cancelled
records were identified.

These records are retained because cancellation information is required
for analysing cancellation impact and profit leakage. A
Cancellation_Flag is added so that cancelled transactions can be
identified separately during downstream analysis.

In [53]:
sales_clean["Cancellation_Flag"]="No"
sales_clean.loc[sales_clean["Order_Status"]=="Cancelled","Cancellation_Flag"]="Yes"
sales_clean["Cancellation_Flag"].value_counts()

Cancellation_Flag
No     74724
Yes     5659
Name: count, dtype: int64

### Cancelled Orders Appearing in Sales Result

#### After removing exact duplicate records, the cleaned Sales Transactions table contains 5,659 cancelled records. These records were retained and identified using the Cancellation_Flag column.

#### The original Order_Status values were preserved so that cancelled orders can be analysed separately from completed sales and profit calculations.

### Cleaning step 9: Final Date Conversion

All date columns are converted to proper datetime data types before the
cleaned dataset is exported.

The dates are kept as actual date values rather than converted to text.
This allows the cleaned data to be loaded correctly into SQL Server
using the DATE datatype.

In [54]:
# convert remaining date columns to actual Date dtype
sales_clean["Order_Date"]=pd.to_datetime(sales_clean["Order_Date"],errors="coerce")
inventory_clean["Date"]=pd.to_datetime(inventory_clean["Date"],errors="coerce")
customer_clean["Signup_Date"]=pd.to_datetime(customer_clean["Signup_Date"],errors="coerce")
promotions_clean["Start_Date"]=pd.to_datetime(promotions_clean["Start_Date"],errors="coerce")
promotions_clean["End_Date"]=pd.to_datetime(promotions_clean["End_Date"],errors="coerce")
returns_clean["Return_Date"]=pd.to_datetime(returns_clean["Return_Date"],errors="coerce")

In [55]:
# because Order_Date has already been converted, we won't see the original invalid values.
# So we need to compare against the raw sales DataFrame
raw_order_dates = pd.to_datetime(sales["Order_Date"],errors="coerce")

sales.loc[raw_order_dates.isna(),["Order_ID", "Order_Date"]].head(20)

,Order_ID,Order_Date
47,ORD1000047,32/13/2025
112,ORD1000112,32/13/2025
427,ORD1000427,32/13/2025
447,ORD1000447,32/13/2025
707,ORD1000707,32/13/2025
965,ORD1000965,32/13/2025
1020,ORD1001020,32/13/2025
1109,ORD1001109,32/13/2025
1379,ORD1001379,32/13/2025
1522,ORD1001522,32/13/2025


In [56]:
# we now need to check whether the same situation exists in the other date columns.
date_columns_raw = {"Sales_Transactions": ("sales", "Order_Date"),
    "Inventory_Daily": ("inventory", "Date"),
    "Returns": ("returns", "Return_Date"),
    "Vendor_Supply": ("vendor", "PO_Date"),
    "Vendor_Supply_Expected": ("vendor", "Expected_Delivery_Date"),
    "Vendor_Supply_Actual": ("vendor", "Actual_Delivery_Date"),
    "Promotions_Start": ("promotions", "Start_Date"),
    "Promotions_End": ("promotions", "End_Date"),
    "Customer_Master": ("customer", "Signup_Date")}
date_columns_raw

{'Sales_Transactions': ('sales', 'Order_Date'),
 'Inventory_Daily': ('inventory', 'Date'),
 'Returns': ('returns', 'Return_Date'),
 'Vendor_Supply': ('vendor', 'PO_Date'),
 'Vendor_Supply_Expected': ('vendor', 'Expected_Delivery_Date'),
 'Vendor_Supply_Actual': ('vendor', 'Actual_Delivery_Date'),
 'Promotions_Start': ('promotions', 'Start_Date'),
 'Promotions_End': ('promotions', 'End_Date'),
 'Customer_Master': ('customer', 'Signup_Date')}

In [57]:
dataframes = {"sales": sales,
    "inventory": inventory,
    "returns": returns,
    "vendor": vendor,
    "promotions": promotions,
    "customer": customer}

In [58]:
for name, (df_name, column) in date_columns_raw.items():
    df = dataframes[df_name]  
    raw_dates = pd.to_datetime(df[column],errors="coerce" )
    invalid_count = raw_dates.isna().sum()
    print(f"{name} - Invalid/Missing dates: {invalid_count}")

Sales_Transactions - Invalid/Missing dates: 612
Inventory_Daily - Invalid/Missing dates: 505
Returns - Invalid/Missing dates: 47
Vendor_Supply - Invalid/Missing dates: 228
Vendor_Supply_Expected - Invalid/Missing dates: 228
Vendor_Supply_Actual - Invalid/Missing dates: 228
Promotions_Start - Invalid/Missing dates: 2
Promotions_End - Invalid/Missing dates: 2
Customer_Master - Invalid/Missing dates: 153


### Cleaning action 9 Date conversion result:

#### Invalid date values such as 32/13/2025 were converted to NaT during datetime conversion and retained as missing values for data-quality tracking. No replacement dates were inferred.

### Cleaning step 10: SAME SKU_ID format across all tables

In [59]:
# Check the format of SKU_ID across all tables.
# This helps us identify whether SKU IDs contain hyphens or other formatting differences before applying a common standard.

sku_tables = {
    "Sales_Transactions": sales_clean,
    "Product_Master": product_clean,
    "Inventory_Daily": inventory_clean,
    "Returns": returns_clean,
    "Vendor_Supply": vendor_clean
}

for table_name, df in sku_tables.items():
    sku_values = df["SKU_ID"].astype(str)

    hyphen_count = sku_values.str.contains("-", na=False).sum()
    underscore_count = sku_values.str.contains("_", na=False).sum()

    print(f"\n{table_name}")
    print("Total records:", len(df))
    print("SKU IDs containing '-':", hyphen_count)
    print("SKU IDs containing '_':", underscore_count)


Sales_Transactions
Total records: 80383
SKU IDs containing '-': 1588
SKU IDs containing '_': 0

Product_Master
Total records: 1202
SKU IDs containing '-': 9
SKU IDs containing '_': 0

Inventory_Daily
Total records: 50076
SKU IDs containing '-': 749
SKU IDs containing '_': 0

Returns
Total records: 4617
SKU IDs containing '-': 65
SKU IDs containing '_': 0

Vendor_Supply
Total records: 30085
SKU IDs containing '-': 458
SKU IDs containing '_': 0


In [60]:
# Standardize only the formatting of SKU_ID with .strip() .upper()
# We preserve meaningful characters such as hyphens and underscores.

for df in [sales_clean, product_clean, inventory_clean, returns_clean, vendor_clean]:
    df["SKU_ID"] = (df["SKU_ID"].astype("string").str.strip().str.upper())

In [61]:
# let's check whether SKU_ID itself is unique in cleanProduct_Master.
print("Total Product_Master rows:", len(product_clean))
print("Unique SKU_IDs:", product_clean["SKU_ID"].nunique())

print("Duplicate SKU_ID rows:",product_clean["SKU_ID"].duplicated().sum())

Total Product_Master rows: 1202
Unique SKU_IDs: 1202
Duplicate SKU_ID rows: 0


In [62]:
## For each dataset:
 #How many SKU values do not exist in our 1,202-SKU Product Master?
# query is checking each record's SKU_ID in the other datasets against the official SKU list from Product_Master.
product_skus = set(product_clean["SKU_ID"].dropna())

for name, df in {
    "Sales_Transactions": sales_clean,
    "Inventory_Daily": inventory_clean,
    "Returns": returns_clean,
    "Vendor_Supply": vendor_clean
}.items():

    unmatched = df[df["SKU_ID"].notna() &~df["SKU_ID"].isin(product_skus)]
    print(name, "→ Unmatched SKU rows:", len(unmatched))

Sales_Transactions → Unmatched SKU rows: 2011
Inventory_Daily → Unmatched SKU rows: 1041
Returns → Unmatched SKU rows: 90
Vendor_Supply → Unmatched SKU rows: 646


In [63]:
product_skus = set(product_clean["SKU_ID"].dropna())

for name, df in {
    "Sales_Transactions": sales_clean,
    "Inventory_Daily": inventory_clean,
    "Returns": returns_clean,
    "Vendor_Supply": vendor_clean
}.items():

    unmatched = df[df["SKU_ID"].notna() &~df["SKU_ID"].isin(product_skus)]

    print(f"\n{name}")
    print("Unique unmatched SKUs:", unmatched["SKU_ID"].nunique())
    print("Unmatched rows:", len(unmatched))
    print(unmatched["SKU_ID"].value_counts().head(30))


Sales_Transactions
Unique unmatched SKUs: 1580
Unmatched rows: 2011
SKU_ID
SKU00178     70
SKU00585     64
SKU00154     64
SKU00755     59
SKU01189     59
SKU00160     56
SKU00722     52
SKU-70762     2
SKU-76747     2
SKU-81291     2
SKU-73341     2
SKU-50205     2
SKU-20916     2
SKU-50174     2
SKU-02852     2
SKU-13506     2
SKU-74687     2
SKU-88556     2
SKU-00931     2
SKU-77885     2
SKU-84615     2
SKU-66322     1
SKU-85136     1
SKU-13507     1
SKU-91207     1
SKU-01089     1
SKU-55802     1
SKU-16651     1
SKU-04840     1
SKU-66108     1
Name: count, dtype: Int64

Inventory_Daily
Unique unmatched SKUs: 754
Unmatched rows: 1041
SKU_ID
SKU00755     49
SKU00160     47
SKU00722     45
SKU01189     42
SKU00178     39
SKU00154     36
SKU00585     34
SKU-48167     2
SKU-65993     2
SKU-44207     1
SKU-72109     1
SKU-73550     1
SKU-87657     1
SKU-40691     1
SKU-99186     1
SKU-07717     1
SKU-18525     1
SKU-46574     1
SKU-40245     1
SKU-48351     1
SKU-06619     1
SKU-98337 

### check Inventory

We already have:

1,041 unmatched Inventory rows / 754 unique unmatched SKUs.

In [64]:
inventory_clean["SKU_ID_Original"]=inventory_clean["SKU_ID"]

In [65]:
inventory_unmatched = inventory_clean[
    inventory_clean["SKU_ID"].str.upper().str.strip().notna() &
    ~inventory_clean["SKU_ID"].isin(product_skus)
].copy()
inventory_unmatched

,Date,Store_ID,SKU_ID,Reorder_Level,Opening_Stock,Closing_Stock,Warehouse_Stock,SKU_ID_Original
66,2025-04-02,JAI-05,SKU00722,15,59,55.0,171.0,SKU00722
75,2025-01-01,PUN-05,SKU00160,37,59,57.0,242.0,SKU00160
108,2025-04-26,AHM-04,SKU01189,34,61,59.0,270.0,SKU01189
155,2025-04-28,KOC-02,SKU-05832,16,80,80.0,499.0,SKU-05832
172,2025-03-17,BEN-02,SKU-43931,36,16,14.0,303.0,SKU-43931
...,...,...,...,...,...,...,...,...
50319,2025-01-04,PUN-03,SKU-77648,14,69,65.0,292.0,SKU-77648
50337,2025-03-09,KOC-05,SKU-11580,23,78,76.0,329.0,SKU-11580
50350,2025-06-26,KOC-02,SKU-59125,32,113,110.0,448.0,SKU-59125
50418,2025-06-24,MUM-05,SKU-81921,27,118,111.0,229.0,SKU-81921


In [66]:
# Create the Inventory SKU mapping flag
inventory_clean["SKU_Mapping_Status"] = inventory_clean["SKU_ID"].isin(product_skus).map({True: "Mapped", False: "Unmapped"})
print(inventory_clean["SKU_Mapping_Status"].value_counts())
print("unique unmatched SKU_IDs:",inventory_unmatched["SKU_ID"].nunique())

SKU_Mapping_Status
Mapped      49035
Unmapped     1041
Name: count, dtype: int64
unique unmatched SKU_IDs: 754


### Returns

We previously found:

71 unique unmatched Return SKUs
90 unmatched Return rows. recheck and:

determine how many of those 71 unmatched SKUs are only formatting differences, exactly like we did for Sales and Inventory.

In [67]:
# preserve original sku_id as seperate column in returns
returns_clean["SKU_ID_Original"] = returns_clean["SKU_ID"]
returns_clean["SKU_ID"]=returns_clean["SKU_ID"].str.upper().str.strip()

In [68]:
returns_unmatched = returns_clean[
    returns_clean["SKU_ID"].str.upper().str.strip().notna() &
    ~returns_clean["SKU_ID"].isin(product_skus)
].copy()
returns_unmatched

,Return_ID,Order_ID,SKU_ID,Return_Date,Return_Reason,Refund_Amount,Return_Status,Sales_Mapping_Status,SKU_ID_Original
51,RET200051,ORD1079443,SKU00755,2025-01-22,Late Delivery,233.42,Approved,Matched,SKU00755
170,RET200170,ORD1049762,SKU-49005,2025-02-28,Changed Mind,231.11,Approved,Matched,SKU-49005
329,RET200329,ORD1035776,SKU00178,2025-02-18,Wrong Item,56.00,Approved,Matched,SKU00178
332,RET200332,ORD1034258,SKU-09141,2025-03-17,Damaged,81.26,Approved,Matched,SKU-09141
433,RET200433,ORD1015583,SKU00585,2025-05-22,Quality Issue,61.01,Approved,Matched,SKU00585
...,...,...,...,...,...,...,...,...,...
4420,RET204420,ORD1021840,SKU-15744,2025-02-05,Missing Item,57.92,Approved,Matched,SKU-15744
4422,RET204422,ORD1062027,SKU-05504,2025-01-28,Changed Mind,825.08,Approved,Matched,SKU-05504
4575,RET-3128431,ORD-4086696,SKU-82504,2025-03-23,Wrong Item,192.95,Rejected,Missing,SKU-82504
4616,RET-6813841,ORD-3266370,SKU00585,2025-06-22,Quality Issue,223.06,Approved,Missing,SKU00585


In [69]:
# flagg unmatched return records 
returns_clean["SKU_Mapping_Status"] = returns_clean["SKU_ID"].isin(product_skus).map({True: "Mapped", False: "Unmapped"})
print("unique unmatched SKU_IDs:",returns_unmatched["SKU_ID"].nunique())
print(returns_clean["SKU_Mapping_Status"].value_counts())

unique unmatched SKU_IDs: 71
SKU_Mapping_Status
Mapped      4527
Unmapped      90
Name: count, dtype: int64


### Find possible formatting matches in vendor

In [70]:
# create original skuid column for audit
vendor_clean["SKU_ID_Original"] = vendor_clean["SKU_ID"]

In [71]:
# format sku_id column with strip() and upper()
vendor_clean["SKU_ID"] = vendor_clean["SKU_ID"].str.upper().str.strip()

In [72]:
vendor_unmatched = vendor_clean[vendor_clean["SKU_ID"].str.upper().str.strip().notna() &
    ~vendor_clean["SKU_ID"].isin(product_skus)
].copy()
vendor_unmatched

,PO_ID,Vendor_ID,SKU_ID,Store_ID,PO_Date,Expected_Delivery_Date,Actual_Delivery_Date,Supplied_Quantity,Procurement_Cost,Delay_Days,SLA_Status,SKU_ID_Original
119,PO300119,VEND0019,SKU00178,KOL-04,2025-02-28,2025-03-03,2025-03-05,51,47.42,2.0,Late,SKU00178
145,PO300145,VEND0111,SKU-03781,DEL-02,2025-05-04,2025-05-13,2025-05-13,66,205.23,0.0,On Time,SKU-03781
162,PO300162,VEND0077,SKU-19489,HYD-01,2025-02-20,2025-02-23,2025-02-23,66,42.69,0.0,On Time,SKU-19489
195,PO300195,VEND0063,SKU-04549,MUM-02,2025-06-18,2025-06-22,2025-06-23,49,66.82,1.0,Late,SKU-04549
198,PO300198,VEND0005,SKU00160,CHE-01,2025-04-22,2025-04-27,2025-05-02,66,66.43,5.0,Late,SKU00160
...,...,...,...,...,...,...,...,...,...,...,...,...
30083,PO312173,VEND0071,SKU-01736,KOL-04,2025-03-07,2025-03-12,2025-03-12,65,140.69,0.0,On Time,SKU-01736
30149,PO300821,VEND0095,SKU-64911,JAI-05,2025-03-14,2025-03-18,2025-03-18,53,112.07,0.0,On Time,SKU-64911
30201,PO319934,VEND0039,SKU-93950,DEL-05,2025-03-10,2025-03-19,2025-03-19,59,104.75,0.0,On Time,SKU-93950
30296,PO306794,VEND0103,SKU-12464,JAI-01,2025-05-27,2025-06-04,2025-06-06,72,66.36,2.0,Late,SKU-12464


In [73]:
# flagg unmatched recordds as we did for others
vendor_clean["SKU_Mapping_Status"] = vendor_clean["SKU_ID"].isin(product_skus).map({True: "Mapped", False: "Unmapped"})
print("unique unmatched SKU_IDs:",vendor_unmatched["SKU_ID"].nunique())
print(vendor_clean["SKU_Mapping_Status"].value_counts())

unique unmatched SKU_IDs: 463
SKU_Mapping_Status
Mapped      29439
Unmapped      646
Name: count, dtype: int64


### cleani step 11: null values check

In [74]:
# Calculate the number and percentage of missing values
# for every column in each cleaned dataset.

dataset_for_nulls = {
    "Sales": sales_clean,
    "Product_Master": product_clean,
    "Inventory_Daily": inventory_clean,
    "Returns": returns_clean,
    "Vendor_Supply": vendor_clean,
    "Promotions": promotions,
    "Customer_Master": customer_clean
}

for name , df in dataset_for_nulls.items():
    print(f"\n{name}")
    print("-" * 50)
    null_count= df.isnull().sum()
    null_percent= (null_count/len(df))*100
    null_summary=pd.DataFrame({
        "Null_Count": null_count,
        "Null_Percent": null_percent})
    display(null_summary)


Sales
--------------------------------------------------


,Null_Count,Null_Percent
Order_ID,0,0.000000
Order_Date,612,0.761355
Store_ID,0,0.000000
Customer_ID,0,0.000000
SKU_ID,0,0.000000
Quantity,0,0.000000
Selling_Price,326,0.405558
Discount_Amount,326,0.405558
Promo_ID,53830,66.966896
Delivery_Type,0,0.000000



Product_Master
--------------------------------------------------


,Null_Count,Null_Percent
SKU_ID,0,0.000000
SKU_Name,0,0.000000
Category,0,0.000000
Sub_Category,0,0.000000
Brand,12,0.998336
Tax_Percent,0,0.000000
Unit_Cost,12,0.998336
MRP,0,0.000000



Inventory_Daily
--------------------------------------------------


...

,Null_Count,Null_Percent
PO_ID,0,0.000000
Vendor_ID,0,0.000000
SKU_ID,0,0.000000
Store_ID,0,0.000000
PO_Date,228,0.757853
Expected_Delivery_Date,228,0.757853
Actual_Delivery_Date,228,0.757853
Supplied_Quantity,0,0.000000
Procurement_Cost,152,0.505235
Delay_Days,455,1.512382



Promotions
--------------------------------------------------


,Null_Count,Null_Percent
Promo_ID,0,0.000000
Promo_Type,0,0.000000
Start_Date,0,0.000000
End_Date,0,0.000000
Target_Category,0,0.000000
Promo_Budget,4,1.904762



Customer_Master
--------------------------------------------------


,Null_Count,Null_Percent
Customer_ID,0,0.000000
City,153,1.017558
Signup_Date,153,1.017558
Customer_Type,0,0.000000


In [75]:
# Create a consolidated report containing only columns
# that have missing values across all 7 datasets.

null_report = []

for name, df in dataset_for_nulls.items():
    null_count = df.isnull().sum()
    null_percent = (null_count / len(df)) * 100
    for column in null_count[null_count > 0].index:
        null_report.append({
            "Table": name,
            "Column": column,
            "Null_Count": null_count[column],
            "Null_Percent": null_percent[column],
            "Total_Rows": len(df)})

null_report_df = pd.DataFrame(null_report)
null_report_df = null_report_df.sort_values(["Table", "Column"]).reset_index(drop=True)
display(null_report_df)

,Table,Column,Null_Count,Null_Percent,Total_Rows
0,Customer_Master,City,153,1.017558,15036
1,Customer_Master,Signup_Date,153,1.017558,15036
2,Inventory_Daily,Closing_Stock,252,0.503235,50076
3,Inventory_Daily,Date,505,1.008467,50076
4,Inventory_Daily,Warehouse_Stock,252,0.503235,50076
5,Product_Master,Brand,12,0.998336,1202
6,Product_Master,Unit_Cost,12,0.998336,1202
7,Promotions,Promo_Budget,4,1.904762,210
8,Returns,Return_Date,47,1.017977,4617
9,Returns,Return_Reason,47,1.017977,4617


### cleanig step 11 result:
Retained as NULLs only because filling or deleting them could lead to wrong analysis.

## cleang step 12: Finding Potential Outliers

In [76]:
# Count potential outliers for every numeric column
outliers_summary=[]
for name,df in dataset_for_nulls.items():
    print(f"\n============={name}===========")
    num_columns = df.select_dtypes(include="number").columns
    for column in num_columns:
        q1=df[column].quantile(0.25)
        q3=df[column].quantile(0.75)
        iqr=q3-q1
        lower_bound= q1-1.5*iqr
        upper_bound= q3+1.5*iqr
        outlier_count=((df[column]<lower_bound) | (df[column]>upper_bound)).sum()
        outliers_summary.append({
            "Dataset": name,
            "Column": column,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "Lower_Bound": lower_bound,
            "Upper_Bound": upper_bound,
            "Outlier_Count": outlier_count
        })
outlier_summary_df=pd.DataFrame(outliers_summary)
outlier_summary_df


=============Sales===========

=============Product_Master===========

=============Inventory_Daily===========

=============Returns===========

=============Vendor_Supply===========

=============Promotions===========

=============Customer_Master===========


,Dataset,Column,Q1,Q3,IQR,Lower_Bound,Upper_Bound,Outlier_Count
0,Sales,Quantity,2.000,4.000,2.00,-1.000,7.000,622
1,Sales,Selling_Price,63.570,153.170,89.60,-70.830,287.570,3390
2,Sales,Discount_Amount,12.750,64.560,51.81,-64.965,142.275,5164
3,Product_Master,Tax_Percent,5.000,18.000,13.00,-14.500,37.500,0
4,Product_Master,Unit_Cost,48.255,116.105,67.85,-53.520,217.880,38
5,Product_Master,MRP,73.925,173.465,99.54,-75.385,322.775,55
6,Inventory_Daily,Reorder_Level,17.000,32.000,15.00,-5.500,54.500,0
7,Inventory_Daily,Opening_Stock,30.000,90.000,60.00,-60.000,180.000,0
8,Inventory_Daily,Closing_Stock,28.000,88.000,60.00,-62.000,178.000,0
9,Inventory_Daily,Warehouse_Stock,161.000,387.000,226.00,-178.000,726.000,0


# Final Validation — Cleaned Dataset
After applying the required data-cleaning treatments, the cleaned datasets
are validated before being exported.

The original raw dataset remains unchanged.

In [77]:
# Display the row count of each cleaned table.

cleaned_tables = {"Sales_Transactions": sales_clean,
    "Product_Master": product_clean,
    "Inventory_Daily": inventory_clean,
    "Returns": returns_clean,
    "Vendor_Supply": vendor_clean,
    "Promotions": promotions_clean,
    "Customer_Master": customer_clean}

for table_name, df in cleaned_tables.items():
    print(f"\n{table_name}: {len(df):,} rows")
    print(df.dtypes)


Sales_Transactions: 80,383 rows
Order_ID                          object
Order_Date                datetime64[ns]
Store_ID                          object
Customer_ID                       object
SKU_ID                    string[python]
Quantity                           int64
Selling_Price                    float64
Discount_Amount                  float64
Promo_ID                          object
Delivery_Type                     object
Payment_Mode                      object
Order_Status                      object
Original_SKU_ID                   object
SKU_Mapping_Status                object
Negative_Quantity_Flag            object
Original_Promo_ID                 object
Promo_Mapping_Status              object
Original_Store_ID                 object
Cancellation_Flag                 object
dtype: object

Product_Master: 1,202 rows
SKU_ID          string[python]
SKU_Name                object
Category                object
Sub_Category            object
Brand                 

In [78]:
# Final duplicate validation

for table_name, df in cleaned_tables.items():
    print(f"{table_name}: {df.duplicated().sum():,} duplicate rows")

Sales_Transactions: 0 duplicate rows
Product_Master: 0 duplicate rows
Inventory_Daily: 0 duplicate rows
Returns: 0 duplicate rows
Vendor_Supply: 0 duplicate rows
Promotions: 0 duplicate rows
Customer_Master: 0 duplicate rows


In [79]:
# Validate Sales cleaning flags

print("SKU Mapping Status:")
print(sales_clean["SKU_Mapping_Status"].value_counts())

print("\nNegative Quantity Flag:")
print(sales_clean["Negative_Quantity_Flag"].value_counts())

print("\nPromo Mapping Status:")
print(sales_clean["Promo_Mapping_Status"].value_counts())

print("\nCancellation Flag:")
print(sales_clean["Cancellation_Flag"].value_counts())

SKU Mapping Status:
SKU_Mapping_Status
Matched    78372
Missing     2011
Name: count, dtype: int64

Negative Quantity Flag:
Negative_Quantity_Flag
No     80057
Yes      326
Name: count, dtype: int64

Promo Mapping Status:
Promo_Mapping_Status
No Promotion    50685
Matched         26553
Missing          3145
Name: count, dtype: int64

Cancellation Flag:
Cancellation_Flag
No     74724
Yes     5659
Name: count, dtype: int64


In [80]:
# Validate Returns-to-Sales mapping status

print(returns_clean["Sales_Mapping_Status"].value_counts())

Sales_Mapping_Status
Matched    4467
Missing     150
Name: count, dtype: int64


In [81]:
print(vendor_clean[["PO_Date", "Expected_Delivery_Date", "Actual_Delivery_Date"]].dtypes)

PO_Date                   datetime64[ns]
Expected_Delivery_Date    datetime64[ns]
Actual_Delivery_Date      datetime64[ns]
dtype: object


In [82]:
# Validate vendor SLA classification

print(vendor_clean["SLA_Status"].value_counts())

SLA_Status
On Time    20638
Late        8912
Missing      455
Early         80
Name: count, dtype: int64


In [83]:
# Confirm the standardized Store_ID count

print("Unique standardized Store_IDs:",sales_clean["Store_ID"].nunique())

Unique standardized Store_IDs: 150


### SQL Schema Design — Key and Datatype Validation

Before creating the SQL Server tables, the cleaned datasets are checked
to identify suitable primary keys, text-column lengths, numeric ranges,
and nullable columns.

This ensures that the SQL schema is based on the actual cleaned data.

In [84]:
# Check key uniqueness, text lengths, numeric ranges, and missing values
# before creating the SQL Server schema.

for table_name, df in cleaned_tables.items():
    print(f"\n{table_name}")
    print("-" * 60)

    # 1. Missing values
    print("\nMissing values:")
    print(df.isna().sum()[df.isna().sum() > 0])

    # 2. Unique values for ID columns
    id_columns = [column for column in df.columns
        if "_ID" in column]

    print("\nID column uniqueness:")
    for column in id_columns:
        print(f"{column}: {df[column].nunique()} unique / {len(df)} rows")

    # 3. Maximum text length
    text_columns = df.select_dtypes(include="object").columns

    print("\nMaximum text lengths:")
    for column in text_columns:
        max_length = df[column].dropna().astype(str).str.len().max()
        print(f"{column}: {max_length}")

    # 4. Numeric ranges
    numeric_columns = df.select_dtypes(include=["int64", "float64"]).columns
    print("\nNumeric ranges:")
    for column in numeric_columns:
        print(f"{column}: Min = {df[column].min()}, Max = {df[column].max()}" )
        


Sales_Transactions
------------------------------------------------------------

Missing values:
Order_Date             612
Selling_Price          326
Discount_Amount        326
Promo_ID             53830
Payment_Mode           326
Original_Promo_ID    50685
dtype: int64

ID column uniqueness:
Order_ID: 80000 unique / 80383 rows
Store_ID: 150 unique / 80383 rows
Customer_ID: 14932 unique / 80383 rows
SKU_ID: 2774 unique / 80383 rows
Promo_ID: 195 unique / 80383 rows
Original_SKU_ID: 2774 unique / 80383 rows
Original_Promo_ID: 201 unique / 80383 rows
Original_Store_ID: 462 unique / 80383 rows

Maximum text lengths:
Order_ID: 10
Store_ID: 6
Customer_ID: 10
Promo_ID: 6
Delivery_Type: 13
Payment_Mode: 10
Order_Status: 9
Original_SKU_ID: 9
SKU_Mapping_Status: 7
Negative_Quantity_Flag: 3
Original_Promo_ID: 10
Promo_Mapping_Status: 12
Original_Store_ID: 6
Cancellation_Flag: 3

Numeric ranges:
Quantity: Min = -8, Max = 12
Selling_Price: Min = 10.7, Max = 807.17
Discount_Amount: Min = 0.0, Max

In [85]:
# Investigate repeated identifiers like "__IDs" before defining SQL primary keys.

# selects the Order_ID column from the SalesDataFrame and counts how many times each Order_ID appears.
#then keep only Order_IDs that appear more than once.
print("=== Sales: repeated Order_IDs ===")
print(sales_clean["Order_ID"].value_counts().loc[lambda x: x > 1])

print("\n=== Returns: repeated Return_IDs ===")
print(returns_clean["Return_ID"].value_counts().loc[lambda x: x > 1])

print("\n=== Vendor: repeated PO_IDs ===")
print(vendor_clean["PO_ID"].value_counts().loc[lambda x: x > 1].head(10))

print("\n=== Promotions: repeated Promo_IDs ===")
print(promotions_clean["Promo_ID"].value_counts().loc[lambda x: x > 1])

print("\n=== Customer: repeated Customer_IDs ===")
print(customer_clean["Customer_ID"].value_counts().loc[lambda x: x > 1])

print("\n=== Inventory: repeated Date + Store_ID + SKU_ID combinations ===")

inventory_key = (inventory_clean.groupby(["Date", "Store_ID", "SKU_ID"]).size())

print(inventory_key[inventory_key > 1])

=== Sales: repeated Order_IDs ===
Order_ID
ORD1025084    2
ORD1008146    2
ORD1078109    2
ORD1001920    2
ORD1061841    2
             ..
ORD1070036    2
ORD1010182    2
ORD1039967    2
ORD1010304    2
ORD1000204    2
Name: count, Length: 383, dtype: int64

=== Returns: repeated Return_IDs ===
Return_ID
RET202444    2
RET202481    2
RET202264    2
RET201120    2
RET200894    2
Name: count, dtype: int64

=== Vendor: repeated PO_IDs ===
PO_ID
PO320149    2
PO307329    2
PO300590    2
PO304819    2
PO326433    2
PO306051    2
PO321348    2
PO322920    2
PO301707    2
PO328953    2
Name: count, dtype: int64

=== Promotions: repeated Promo_IDs ===
Promo_ID
PROMO_XXXX    6
PR0200        2
Name: count, dtype: int64

=== Customer: repeated Customer_IDs ===
Customer_ID
CUST009802    2
CUST010540    2
CUST014341    2
CUST010748    2
CUST003962    2
CUST008958    2
CUST010734    2
CUST006624    2
CUST006381    2
CUST000545    2
CUST010151    2
CUST013909    2
CUST009778    2
CUST000324    2
CUST

In [86]:
# Inspect repeated business IDs before deciding SQL primary keys. 1st do for less no.of records dataframes

print("=== RETURNS: repeated Return_IDs ===")

duplicate_return_ids = (returns_clean["Return_ID"].value_counts().loc[lambda x: x > 1].index)
display(returns_clean[returns_clean["Return_ID"].isin(duplicate_return_ids)].sort_values("Return_ID"))

print("\n=== PROMOTIONS: repeated Promo_IDs ===")
duplicate_promo_ids = (promotions_clean["Promo_ID"].value_counts().loc[lambda x: x > 1].index)

display(promotions_clean[promotions_clean["Promo_ID"].isin(duplicate_promo_ids)
    ].sort_values("Promo_ID"))

print("\n=== CUSTOMER: repeated Customer_IDs ===")
duplicate_customer_ids = (customer_clean["Customer_ID"].value_counts().loc[lambda x: x > 1].index)
display(customer_clean[customer_clean["Customer_ID"].isin(duplicate_customer_ids)].sort_values("Customer_ID"))

=== RETURNS: repeated Return_IDs ===


,Return_ID,Order_ID,SKU_ID,Return_Date,Return_Reason,Refund_Amount,Return_Status,Sales_Mapping_Status,SKU_ID_Original,SKU_Mapping_Status
894,RET200894,ORD1016014,SKU00011,NaT,Wrong Item,208.18,Approved,Matched,SKU00011,Mapped
4486,RET200894,ORD1016014,SKU00011,2025-06-02,Wrong Item,208.18,Approved,Matched,SKU00011,Mapped
1120,RET201120,ORD1019996,SKU00855,NaT,Damaged,477.65,Approved,Matched,SKU00855,Mapped
4480,RET201120,ORD1019996,SKU00855,2025-03-25,Damaged,477.65,Approved,Matched,SKU00855,Mapped
2264,RET202264,ORD1077650,SKU00741,2025-04-25,Damaged,661.81,Approved,Matched,SKU00741,Mapped
4506,RET202264,ORD1077650,SKU00741,2025-04-25,NaN,661.81,Approved,Matched,SKU00741,Mapped
2444,RET202444,ORD1024178,SKU00495,NaT,Missing Item,213.51,Approved,Matched,SKU00495,Mapped
4534,RET202444,ORD1024178,SKU00495,2025-02-15,Missing Item,213.51,Approved,Matched,SKU00495,Mapped
2481,RET202481,ORD1011104,SKU-08481,2025-01-26,Damaged,248.19,Approved,Matched,SKU-08481,Unmapped
4530,RET202481,ORD1011104,SKU00810,2025-01-26,Damaged,248.19,Approved,Matched,SKU00810,Mapped



=== PROMOTIONS: repeated Promo_IDs ===


,Promo_ID,Promo_Type,Start_Date,End_Date,Target_Category,Promo_Budget
199,PR0200,Member Exclusive,2025-01-27,2025-02-15,Beverages,NaN
206,PR0200,Member Exclusive,2025-01-27,2025-02-15,Beverages,90374.0
66,PROMO_XXXX,BOGO,2025-06-27,2025-07-21,Dairy,290509.0
112,PROMO_XXXX,Member Exclusive,2025-01-18,2025-01-26,Beverages,748977.0
129,PROMO_XXXX,BOGO,2025-03-31,2025-04-27,Snacks,592547.0
132,PROMO_XXXX,Flat %,2025-02-27,2025-03-15,Personal Care,732671.0
141,PROMO_XXXX,BOGO,2025-05-09,2025-05-19,Home Care,650928.0
182,PROMO_XXXX,Bundle,2025-04-02,2025-04-13,Baby Care,605016.0



=== CUSTOMER: repeated Customer_IDs ===


,Customer_ID,City,Signup_Date,Customer_Type
323,CUST000324,Delhi,2025-02-06,Returning
15105,CUST000324,Delhi,NaT,Returning
15248,CUST000545,Ahmedabad,2025-06-13,Returning
544,CUST000545,NaN,2025-06-13,Returning
15008,CUST002233,NaN,2025-06-23,New
2232,CUST002233,Bengaluru,2025-06-23,New
15137,CUST003962,Chennai,2025-01-08,Returning
3961,CUST003962,Chennai,NaT,Returning
6380,CUST006381,NaN,2025-06-21,New
15049,CUST006381,Kolkata,2025-06-21,New


In [87]:
# Check whether repeated records contain the same information except for missing values.
# For each repeated ID, are the repeated records actually the same in all non-missing fields, or do they contain different information?
def check_partial_duplicates(df, id_column):
    
    # .index used to take only actual id values and not count values. store all repeated ids in duplicated_ids
    duplicate_ids = (df[id_column].value_counts().loc[lambda x: x > 1].index)
    
    # exampl- promo1,promo2 of promo_id column are in duplicated_id then .isin() gives True,True and all such rows stores in duplicate_rows
    duplicate_rows = df[df[id_column].isin(duplicate_ids)]
    print(f"\n{ id_column }")

    # loop through each promo_id in duplicated_ids
    for key in duplicate_ids:
        # give me whole row from duplicated_rows for that promo_id column value(1st loop-promo1) == current promo_id(1st loop-promo1)
        # so group will have all records of those repeated '_ids'
        group = duplicate_rows[duplicate_rows[id_column] == key] 
        
        # create empty dict
        differences = {}

        # Loop through every column
        for column in group.columns:     
            values = group[column].dropna().astype(str).unique()       
            if len(values) > 1:
                differences[column] = values    
        print(f"\n{key}")     
        if differences:
            print("Different non-missing values.")
        else:
            print("Same non-missing values — records differ only by missing values.")

# Calling the function with dataframes and id columns
check_partial_duplicates(promotions_clean, "Promo_ID")
check_partial_duplicates(returns_clean, "Return_ID")
check_partial_duplicates(customer_clean, "Customer_ID")


Promo_ID

PROMO_XXXX
Different non-missing values.

PR0200
Same non-missing values — records differ only by missing values.

Return_ID

RET202444
Same non-missing values — records differ only by missing values.

RET202481
Different non-missing values.

RET202264
Same non-missing values — records differ only by missing values.

RET201120
Same non-missing values — records differ only by missing values.

RET200894
Same non-missing values — records differ only by missing values.

Customer_ID

CUST009802
Same non-missing values — records differ only by missing values.

CUST010540
Same non-missing values — records differ only by missing values.

CUST014341
Same non-missing values — records differ only by missing values.

CUST010748
Same non-missing values — records differ only by missing values.

CUST003962
Same non-missing values — records differ only by missing values.

CUST008958
Same non-missing values — records differ only by missing values.

CUST010734
Same non-missing values — record

In [88]:
promotions_clean[promotions_clean["Promo_ID"].isin(["PR0200", "PROMO_XXXX"])].sort_values("Promo_ID")

,Promo_ID,Promo_Type,Start_Date,End_Date,Target_Category,Promo_Budget
199,PR0200,Member Exclusive,2025-01-27,2025-02-15,Beverages,NaN
206,PR0200,Member Exclusive,2025-01-27,2025-02-15,Beverages,90374.0
66,PROMO_XXXX,BOGO,2025-06-27,2025-07-21,Dairy,290509.0
112,PROMO_XXXX,Member Exclusive,2025-01-18,2025-01-26,Beverages,748977.0
129,PROMO_XXXX,BOGO,2025-03-31,2025-04-27,Snacks,592547.0
132,PROMO_XXXX,Flat %,2025-02-27,2025-03-15,Personal Care,732671.0
141,PROMO_XXXX,BOGO,2025-05-09,2025-05-19,Home Care,650928.0
182,PROMO_XXXX,Bundle,2025-04-02,2025-04-13,Baby Care,605016.0


In [89]:
# Check whether PR0200 has any other differences
promotions_clean[promotions_clean["Promo_ID"] == "PR0200"]

,Promo_ID,Promo_Type,Start_Date,End_Date,Target_Category,Promo_Budget
199,PR0200,Member Exclusive,2025-01-27,2025-02-15,Beverages,NaN
206,PR0200,Member Exclusive,2025-01-27,2025-02-15,Beverages,90374.0


### Handling Duplicate Promotion Record — PR0200

PR0200 appears twice with identical promotion details. The only difference
is that one record has a missing Promo_Budget while the other contains a
valid budget value.

The incomplete duplicate is removed and the complete record is retained.

In [90]:
# Remove the incomplete duplicate record for PR0200.
# Keep the record that contains the Promo_Budget value.

# filter with Promo_ID = PR0200 AND "Promo_Budget is missing"
# .drop() removes rows using their index. so .index gives its index number
# removes that index row in promotions_clean and asssign result to itself

promotions_clean = promotions_clean.drop(promotions_clean[(promotions_clean["Promo_ID"] == "PR0200") &
        (promotions_clean["Promo_Budget"].isna())].index)

In [91]:
# Before deciding the exact cleaning treatment for PROMO_XXXX, let's inspect whether PROMO_XXXX appears in the Sales table
sales_clean[sales_clean["Original_Promo_ID"] == "PROMO_XXXX"].shape

(0, 19)

In [92]:
# checking Promo_XXXX having any records in SalesData
sales_clean[sales_clean["Promo_ID"] == "PROMO_XXXX"]
# no sales record found with that promo_id

,Order_ID,Order_Date,Store_ID,Customer_ID,SKU_ID,Quantity,Selling_Price,Discount_Amount,Promo_ID,Delivery_Type,Payment_Mode,Order_Status,Original_SKU_ID,SKU_Mapping_Status,Negative_Quantity_Flag,Original_Promo_ID,Promo_Mapping_Status,Original_Store_ID,Cancellation_Flag


### Handling Ambiguous Promotion ID — PROMO_XXXX

PROMO_XXXX appears in multiple promotion records with different
promotion details. Therefore, the records are not duplicates.

The ID is treated as an ambiguous promotion identifier and the records
are retained because they contain distinct promotion information.
No sales transactions reference this ID.

In [93]:
# Flag PROMO_XXXX as an ambiguous promotion identifier.
# The underlying promotion records are retained.

promotions_clean["Promo_ID_Status"] = "Valid"

promotions_clean.loc[promotions_clean["Promo_ID"] == "PROMO_XXXX","Promo_ID_Status"] = "Ambiguous"

In [94]:
promotions_clean[promotions_clean["Promo_ID"] == "PROMO_XXXX"]

,Promo_ID,Promo_Type,Start_Date,End_Date,Target_Category,Promo_Budget,Promo_ID_Status
66,PROMO_XXXX,BOGO,2025-06-27,2025-07-21,Dairy,290509.0,Ambiguous
112,PROMO_XXXX,Member Exclusive,2025-01-18,2025-01-26,Beverages,748977.0,Ambiguous
129,PROMO_XXXX,BOGO,2025-03-31,2025-04-27,Snacks,592547.0,Ambiguous
132,PROMO_XXXX,Flat %,2025-02-27,2025-03-15,Personal Care,732671.0,Ambiguous
141,PROMO_XXXX,BOGO,2025-05-09,2025-05-19,Home Care,650928.0,Ambiguous
182,PROMO_XXXX,Bundle,2025-04-02,2025-04-13,Baby Care,605016.0,Ambiguous


### inventory_clean primary key check

In [95]:
# whether the combination of Date + Store_ID + SKU_ID is repeated in inventory_clean.
# subset here is only look at these three columns seperatly and detect duplicates with that combination
# defaultly duplicated treats 2nd row as duplicate but by keep=False,treats allrows as duplicates and we want to see those duplicate combination rows.

inventory_clean[inventory_clean.duplicated(subset=["Date", "Store_ID", "SKU_ID"],keep=False)].sort_values(["Date", "Store_ID", "SKU_ID"])

,Date,Store_ID,SKU_ID,Reorder_Level,Opening_Stock,Closing_Stock,Warehouse_Stock,SKU_ID_Original,SKU_Mapping_Status
20116,2025-01-06,DEL-02,SKU00927,36,46,40.0,403.0,SKU00927,Mapped
35560,2025-01-06,DEL-02,SKU00927,26,15,17.0,117.0,SKU00927,Mapped
20104,2025-01-06,KOC-02,SKU00812,29,11,10.0,106.0,SKU00812,Mapped
41672,2025-01-06,KOC-02,SKU00812,38,47,44.0,313.0,SKU00812,Mapped
44937,2025-01-07,HYD-03,SKU00635,16,84,80.0,NaN,SKU00635,Mapped
...,...,...,...,...,...,...,...,...,...
36209,2025-06-28,JAI-02,SKU01095,18,41,39.0,423.0,SKU01095,Mapped
1199,2025-06-30,JAI-02,SKU00624,13,59,57.0,254.0,SKU00624,Mapped
36801,2025-06-30,JAI-02,SKU00624,10,44,37.0,437.0,SKU00624,Mapped
11922,NaT,KOC-05,SKU00380,18,13,12.0,488.0,SKU00380,Mapped


In [96]:
# Let's find out whether these 184 rows represent 92 duplicated combinations appearing exactly twice, 
# or whether some combinations appear 3+ times.
# Without dropna=False, Pandas normally excludes groups(stated combinations) containing missing values.
# size() counts how many rows are in each group.
# id_counts=Each Date + Store + SKU combination and how many times it occurs.
id_counts = (inventory_clean.groupby(["Date", "Store_ID", "SKU_ID"], dropna=False).size())

print("Total repeated combinations rows:", (id_counts > 1).sum())
print("\nFrequency of repeated combinations:")
print(id_counts[id_counts > 1].value_counts().sort_index())

Total repeated combinations rows: 92

Frequency of repeated combinations:
2    92
Name: count, dtype: int64


In [97]:
# 92 unique Date + Store_ID + SKU_ID combinations are repeated
# we should NOT delete these 184 rows as duplicates. They are not exact duplicates, and deleting one could remove legitimate inventory information.
# Next: understand whether there is another column that distinguishes them
inventory_repeated = inventory_clean[inventory_clean.duplicated(subset=["Date", "Store_ID", "SKU_ID"],keep=False)
].sort_values(["Date", "Store_ID", "SKU_ID"])

display(inventory_repeated)

,Date,Store_ID,SKU_ID,Reorder_Level,Opening_Stock,Closing_Stock,Warehouse_Stock,SKU_ID_Original,SKU_Mapping_Status
20116,2025-01-06,DEL-02,SKU00927,36,46,40.0,403.0,SKU00927,Mapped
35560,2025-01-06,DEL-02,SKU00927,26,15,17.0,117.0,SKU00927,Mapped
20104,2025-01-06,KOC-02,SKU00812,29,11,10.0,106.0,SKU00812,Mapped
41672,2025-01-06,KOC-02,SKU00812,38,47,44.0,313.0,SKU00812,Mapped
44937,2025-01-07,HYD-03,SKU00635,16,84,80.0,NaN,SKU00635,Mapped
...,...,...,...,...,...,...,...,...,...
36209,2025-06-28,JAI-02,SKU01095,18,41,39.0,423.0,SKU01095,Mapped
1199,2025-06-30,JAI-02,SKU00624,13,59,57.0,254.0,SKU00624,Mapped
36801,2025-06-30,JAI-02,SKU00624,10,44,37.0,437.0,SKU00624,Mapped
11922,NaT,KOC-05,SKU00380,18,13,12.0,488.0,SKU00380,Mapped


#### Cleaning Inventory_clean Primary Key result:
#### We can create a surrogate primary key IE, IDENTITY KEY TO UNIQULY IDENTIFY EACH ROW

## returns_clean PRIMARY KEY

In [98]:
# Check whether Return_ID values are unique.
# If no Return_ID is repeated, Return_ID can be used as the
# primary key candidate in SQL.

return_id_counts = returns_clean["Return_ID"].value_counts()

repeated_return_ids = return_id_counts[return_id_counts > 1]

print("Number of repeated Return_IDs:", len(repeated_return_ids))
print("\nRepeated Return_IDs:")
print(repeated_return_ids)

Number of repeated Return_IDs: 5

Repeated Return_IDs:
Return_ID
RET202444    2
RET202481    2
RET202264    2
RET201120    2
RET200894    2
Name: count, dtype: int64


In [99]:
# Inspect the repeated Return_ID records
# Display all records having repeated Return_ID values.
# We will compare the other columns to determine the correct SQL key.

repeated_returns = returns_clean[returns_clean["Return_ID"].isin(repeated_return_ids.index)
].sort_values("Return_ID")

display(repeated_returns)

,Return_ID,Order_ID,SKU_ID,Return_Date,Return_Reason,Refund_Amount,Return_Status,Sales_Mapping_Status,SKU_ID_Original,SKU_Mapping_Status
894,RET200894,ORD1016014,SKU00011,NaT,Wrong Item,208.18,Approved,Matched,SKU00011,Mapped
4486,RET200894,ORD1016014,SKU00011,2025-06-02,Wrong Item,208.18,Approved,Matched,SKU00011,Mapped
1120,RET201120,ORD1019996,SKU00855,NaT,Damaged,477.65,Approved,Matched,SKU00855,Mapped
4480,RET201120,ORD1019996,SKU00855,2025-03-25,Damaged,477.65,Approved,Matched,SKU00855,Mapped
2264,RET202264,ORD1077650,SKU00741,2025-04-25,Damaged,661.81,Approved,Matched,SKU00741,Mapped
4506,RET202264,ORD1077650,SKU00741,2025-04-25,NaN,661.81,Approved,Matched,SKU00741,Mapped
2444,RET202444,ORD1024178,SKU00495,NaT,Missing Item,213.51,Approved,Matched,SKU00495,Mapped
4534,RET202444,ORD1024178,SKU00495,2025-02-15,Missing Item,213.51,Approved,Matched,SKU00495,Mapped
2481,RET202481,ORD1011104,SKU-08481,2025-01-26,Damaged,248.19,Approved,Matched,SKU-08481,Unmapped
4530,RET202481,ORD1011104,SKU00810,2025-01-26,Damaged,248.19,Approved,Matched,SKU00810,Mapped


In [100]:
## Return_ID + SKU_ID combination repeated records 
# Check whether Return_ID + SKU_ID uniquely identifies each return record.

return_key_counts = returns_clean.value_counts(subset=["Return_ID", "SKU_ID"])

repeated_return_keys = return_key_counts[return_key_counts > 1]

print("Repeated Return_ID + SKU_ID combinations:",len(repeated_return_keys))

print("\nRepeated combinations:")
print(repeated_return_keys)

Repeated Return_ID + SKU_ID combinations: 4

Repeated combinations:
Return_ID  SKU_ID  
RET200894  SKU00011    2
RET202264  SKU00741    2
RET202444  SKU00495    2
RET201120  SKU00855    2
Name: count, dtype: int64


In [101]:
# Validate that Return_ID + SKU_ID is now unique

return_key_counts = returns_clean.value_counts(
    subset=["Return_ID", "SKU_ID"])

repeated_return_keys = return_key_counts[return_key_counts > 1]

print("Repeated Return_ID + SKU_ID combinations:",len(repeated_return_keys))

Repeated Return_ID + SKU_ID combinations: 4


### cleaning action 11 result:
#### keep the complete record and remove the incomplete duplicate. so Return_ID + SKU_ID combination can be PK

## Vendor_Supply Primary key investigation:

We'll now investigate only the key/business-grain of Vendor_Supply.

let's check whether PO_ID can be the primary key.

In [102]:
# Check whether PO_ID uniquely identifies each vendor supply record

po_id_counts = vendor_clean["PO_ID"].value_counts()

repeated_po_ids = po_id_counts[po_id_counts > 1]

print("Repeated PO_IDs:", len(repeated_po_ids))
print("\nRepeated PO_ID values:")
print(repeated_po_ids)

Repeated PO_IDs: 85

Repeated PO_ID values:
PO_ID
PO320149    2
PO307329    2
PO300590    2
PO304819    2
PO326433    2
           ..
PO300253    2
PO321297    2
PO313118    2
PO311559    2
PO315003    2
Name: count, Length: 85, dtype: int64


In [103]:
# A purchase order can potentially contain multiple SKU/store/vendor combinations, 
# so let's first inspect whether PO_ID + SKU_ID + Store_ID uniquely identifies each record.
vendor_key_counts = vendor_clean.value_counts(subset=["PO_ID", "SKU_ID", "Store_ID"])
vendor_key_counts

PO_ID     SKU_ID    Store_ID
PO318056  SKU00417  JAI-01      2
PO303985  SKU00882  MUM-05      2
PO316517  SKU00289  JAI-05      2
PO310802  SKU00506  CHE-05      2
PO328953  SKU00846  AHM-02      2
                               ..
PO310003  SKU00101  HYD-01      1
PO310002  SKU00255  KOL-04      1
PO310001  SKU00987  KOL-04      1
PO310000  SKU00151  CHE-03      1
PO329999  SKU00503  KOC-05      1
Name: count, Length: 30062, dtype: int64

In [104]:
repeated_vendor_keys = vendor_key_counts[vendor_key_counts > 1]

print("Repeated PO_ID + SKU_ID + Store_ID combinations:",len(repeated_vendor_keys))

print("\nRepeated combinations:")
print(repeated_vendor_keys)

Repeated PO_ID + SKU_ID + Store_ID combinations: 23

Repeated combinations:
PO_ID     SKU_ID    Store_ID
PO318056  SKU00417  JAI-01      2
PO303985  SKU00882  MUM-05      2
PO316517  SKU00289  JAI-05      2
PO310802  SKU00506  CHE-05      2
PO328953  SKU00846  AHM-02      2
PO309534  SKU00520  PUN-03      2
PO309353  SKU00212  KOC-01      2
PO324092  SKU00384  CHE-05      2
PO322409  SKU00410  KOC-03      2
PO312114  SKU00187  JAI-03      2
PO306051  SKU01019  JAI-03      2
PO309578  SKU00033  JAI-05      2
PO325143  SKU00937  KOL-01      2
PO308287  SKU00811  AHM-04      2
PO328174  SKU00721  HYD-01      2
PO328666  SKU01043  KOC-02      2
PO322803  SKU00676  DEL-04      2
PO318034  SKU00526  AHM-05      2
PO309455  SKU00837  AHM-02      2
PO304103  SKU00420  DEL-02      2
PO300590  SKU00360  KOL-01      2
PO301195  SKU00724  CHE-01      2
PO300694  SKU00848  DEL-02      2
Name: count, dtype: int64


In [105]:
# inspect the 23 repeated records
vendor_key_issues = vendor_clean[vendor_clean.set_index(["PO_ID", "SKU_ID", "Store_ID"]).index.isin(repeated_vendor_keys.index)
].sort_values(["PO_ID", "SKU_ID", "Store_ID"])

vendor_key_issues

,PO_ID,Vendor_ID,SKU_ID,Store_ID,PO_Date,Expected_Delivery_Date,Actual_Delivery_Date,Supplied_Quantity,Procurement_Cost,Delay_Days,SLA_Status,SKU_ID_Original,SKU_Mapping_Status
590,PO300590,VEND0018,SKU00360,KOL-01,2025-05-07,2025-05-15,2025-05-15,61,NaN,0.0,On Time,SKU00360,Mapped
30160,PO300590,VEND0018,SKU00360,KOL-01,2025-05-07,2025-05-15,2025-05-15,61,121.37,0.0,On Time,SKU00360,Mapped
694,PO300694,VEND0114,SKU00848,DEL-02,2025-06-21,2025-06-27,2025-06-27,59,136.48,0.0,On Time,SKU00848,Mapped
30079,PO300694,VEND0114,SKU00848,DEL-02,2025-06-21,2025-06-27,2025-06-27,59,NaN,0.0,On Time,SKU00848,Mapped
1195,PO301195,VEND0104,SKU00724,CHE-01,2025-06-06,2025-06-08,2025-06-08,54,75.68,0.0,On Time,SKU00724,Mapped
30128,PO301195,VEND0104,SKU00724,CHE-01,2025-06-06,2025-06-08,2025-06-08,54,NaN,0.0,On Time,SKU00724,Mapped
3985,PO303985,VEND0093,SKU00882,MUM-05,2025-05-29,2025-05-31,2025-01-06,62,137.99,-145.0,Early,SKU00882,Mapped
30315,PO303985,VEND0093,SKU00882,MUM-05,2025-05-29,2025-05-31,2025-06-01,62,137.99,1.0,Late,SKU00882,Mapped
4103,PO304103,VEND0005,SKU00420,DEL-02,2025-06-24,2025-07-02,2025-07-02,70,90.95,0.0,On Time,SKU00420,Mapped
30095,PO304103,VEND0005,SKU00420,DEL-02,2025-06-24,2025-07-02,2025-02-07,70,90.95,-145.0,Early,SKU00420,Mapped


In [106]:
# check PO300590, PO300694, PO301195, PO306051, PO308287, PO312114, PO316517, PO318034, PO318056, PO322409, 
# PO322803, PO324092, PO325143, PO328174, PO328953 for one value missing duplicates

# check PO303985, PO304103, PO309353, PO309534, PO309455, PO309578, PO310802, PO328666 for date errorness duplicate

- Look here, the records are purely Data Quality issues with incomple-duplicates information.
- PO_IDs with PO300590, PO300694, PO301195, PO306051, PO308287, PO312114, PO316517, PO318034, PO318056, PO322409, PO322803, PO324092, PO325143, PO328174, PO328953 for one column value missing duplicates
- PO_IDs WITH PO303985, PO304103, PO309353, PO309534, PO309455, PO309578, PO310802, PO328666 for date errorness duplicate

The next step should therefore be cleaning the clearly incomplete Vendor duplicates,while preserving conflicting records and documenting them as a data-quality issue.

In [107]:

vendor_clean["Data_Quality_Flag"] = (((vendor_clean["Actual_Delivery_Date"] < vendor_clean["PO_Date"]) |
                                     (vendor_clean["Actual_Delivery_Date"] < vendor_clean["Expected_Delivery_Date"])).
    map({True: "ImpossibleDateCombo", False: "Normal"}))
vendor_clean["Data_Quality_Flag"].value_counts()


Data_Quality_Flag
Normal                 29944
ImpossibleDateCombo      141
Name: count, dtype: int64

In [108]:
# we're using row indices each PO_ID has two records. We want to remove only the incomplete copy and retain the complete copy.
rows_to_delete = [590, 30079, 30128, 6051, 8287, 12114, 30271, 30287, 18056, 30106, 30031, 30231, 30125, 30272, 28953]
# .drop uses indexes to delete rows
vendor_clean = vendor_clean.drop(index=rows_to_delete)

# You should now see one record for each of those 15 PO_IDs, rather than two.
vendor_clean[vendor_clean["PO_ID"].isin
    (["PO300590", "PO300694", "PO301195", "PO306051", "PO308287", "PO312114",
      "PO316517", "PO318034", "PO318056", "PO322409", "PO322803", "PO324092", "PO325143", "PO328174", "PO328953"])]

,PO_ID,Vendor_ID,SKU_ID,Store_ID,PO_Date,Expected_Delivery_Date,Actual_Delivery_Date,Supplied_Quantity,Procurement_Cost,Delay_Days,SLA_Status,SKU_ID_Original,SKU_Mapping_Status,Data_Quality_Flag
694,PO300694,VEND0114,SKU00848,DEL-02,2025-06-21,2025-06-27,2025-06-27,59,136.48,0.0,On Time,SKU00848,Mapped,Normal
1195,PO301195,VEND0104,SKU00724,CHE-01,2025-06-06,2025-06-08,2025-06-08,54,75.68,0.0,On Time,SKU00724,Mapped,Normal
16517,PO316517,VEND0014,SKU00289,JAI-05,2025-04-24,2025-05-03,2025-05-03,50,38.88,0.0,On Time,SKU00289,Mapped,Normal
18034,PO318034,VEND0004,SKU00526,AHM-05,2025-01-01,2025-01-08,2025-01-09,64,61.81,1.0,Late,SKU00526,Mapped,Normal
22409,PO322409,VEND0103,SKU00410,KOC-03,2025-05-26,2025-06-02,2025-06-02,49,265.51,0.0,On Time,SKU00410,Mapped,Normal
22803,PO322803,VEND0066,SKU00676,DEL-04,2025-01-21,2025-01-26,2025-01-31,55,93.00,5.0,Late,SKU00676,Mapped,Normal
24092,PO324092,VEND0044,SKU00384,CHE-05,2025-05-10,2025-05-13,2025-05-14,55,165.61,1.0,Late,SKU00384,Mapped,Normal
25143,PO325143,VEND0027,SKU00937,KOL-01,2025-04-27,2025-05-01,2025-05-01,55,66.25,0.0,On Time,SKU00937,Mapped,Normal
28174,PO328174,VEND0089,SKU00721,HYD-01,2025-05-05,2025-05-13,2025-05-13,49,48.07,0.0,On Time,SKU00721,Mapped,Normal
30136,PO312114,VEND0030,SKU00187,JAI-03,2025-02-17,2025-02-26,2025-02-26,47,91.16,0.0,On Time,SKU00187,Mapped,Normal


### Cleaning action 12 result: 
#### The incomplete i.e, one value missing duplicate records were removed, while the complete corresponding records were retained. 

#### And Impossible / Erroneous Date Combinations – Flagged. The PO_IDs contained conflicting date records, including cases where the actual delivery date occurred before the purchase order date.

#### so the combination duplicates are still there and we can add Identity Key as PK if wanted

## Customer_Master

In [109]:
# Check whether Customer_ID uniquely identifies each customer

customer_id_counts = customer_clean["Customer_ID"].value_counts()

repeated_customer_ids = customer_id_counts[customer_id_counts > 1]

print("Repeated Customer_IDs:", len(repeated_customer_ids))
print("\nRepeated Customer_ID values:")
print(repeated_customer_ids)

Repeated Customer_IDs: 20

Repeated Customer_ID values:
Customer_ID
CUST009802    2
CUST010540    2
CUST014341    2
CUST010748    2
CUST003962    2
CUST008958    2
CUST010734    2
CUST006624    2
CUST006381    2
CUST000545    2
CUST010151    2
CUST013909    2
CUST009778    2
CUST000324    2
CUST012342    2
CUST002233    2
CUST011854    2
CUST008235    2
CUST009738    2
CUST014080    2
Name: count, dtype: int64


In [110]:
# inspect the repeated IDs

customer_key_issues = customer_clean[customer_clean["Customer_ID"].isin(repeated_customer_ids.index)
].sort_values("Customer_ID")

customer_key_issues

,Customer_ID,City,Signup_Date,Customer_Type
323,CUST000324,Delhi,2025-02-06,Returning
15105,CUST000324,Delhi,NaT,Returning
15248,CUST000545,Ahmedabad,2025-06-13,Returning
544,CUST000545,NaN,2025-06-13,Returning
15008,CUST002233,NaN,2025-06-23,New
2232,CUST002233,Bengaluru,2025-06-23,New
15137,CUST003962,Chennai,2025-01-08,Returning
3961,CUST003962,Chennai,NaT,Returning
6380,CUST006381,NaN,2025-06-21,New
15049,CUST006381,Kolkata,2025-06-21,New


In [111]:
# Keep the complete record and remove the incomplete duplicate.
# We should retain conflicting records and flag them

# use the repeated Customer records we already found. and get incomple duplicates
# exclude the 3 conflicting Customer IDs.
customer_incomplete = (customer_key_issues[~customer_key_issues["Customer_ID"].isin
    (["CUST006624", "CUST008235", "CUST009738"]) & 
    # at least one missing value.
    customer_key_issues.isna().any(axis=1)])
customer_incomplete

,Customer_ID,City,Signup_Date,Customer_Type
15105,CUST000324,Delhi,NaT,Returning
544,CUST000545,NaN,2025-06-13,Returning
15008,CUST002233,NaN,2025-06-23,New
3961,CUST003962,Chennai,NaT,Returning
6380,CUST006381,NaN,2025-06-21,New
8957,CUST008958,Chennai,NaT,VIP
15022,CUST009778,NaN,2025-06-01,Returning
15097,CUST009802,NaN,2025-06-10,Returning
15033,CUST010151,NaN,2025-06-28,New
10539,CUST010540,NaN,2025-01-11,New


In [112]:
# Remove incomplete duplicate customer records.
# The complete version of each Customer_ID is retained.

customer_clean = (customer_clean.drop(index=
            [15105, 544, 15008, 3961, 6380, 8957, 15022, 15097, 15033, 10539, 10733, 15169, 15098, 12341, 13908, 14079, 15007]))

In [113]:
# verify the 3 conflicting Customer_IDs remain
customer_clean[customer_clean["Customer_ID"].isin(["CUST006624", "CUST008235", "CUST009738"])]

,Customer_ID,City,Signup_Date,Customer_Type
6623,CUST006624,Delhi,2025-03-05,New
8234,CUST008235,Bengaluru,2025-09-03,Returning
9737,CUST009738,Mumbai,2025-12-01,New
15012,CUST008235,Bengaluru,2025-03-09,Returning
15089,CUST006624,Delhi,2025-05-03,New
15211,CUST009738,Mumbai,2025-01-12,New


In [114]:
# Check what repeated IDs remain
customer_id_counts = customer_clean["Customer_ID"].value_counts()

repeated_customer_ids = customer_id_counts[customer_id_counts > 1]

print("Customer rows after cleaning:", len(customer_clean))
print("Remaining repeated Customer_IDs:", len(repeated_customer_ids))

print("\nRemaining repeated Customer_ID values:")
print(repeated_customer_ids)

Customer rows after cleaning: 15019
Remaining repeated Customer_IDs: 3

Remaining repeated Customer_ID values:
Customer_ID
CUST009738    2
CUST006624    2
CUST008235    2
Name: count, dtype: int64


### cleaning action 13 result is:

#### Incomplete duplicate customer records were removed. Remaining conflicting customer records were retained because the correct source value could not be determined. and customer_id can not be primary key

## Sales_Transactions

In [115]:
# Check Order_ID + SKU_ID + Store_ID uniquely identifies
sales_key_counts = sales_clean.value_counts(subset=["Order_ID", "SKU_ID", "Store_ID"])

repeated_sales_keys = sales_key_counts[sales_key_counts > 1]

print("Repeated Order_ID + SKU_ID + Store_ID combinations:",len(repeated_sales_keys))

print("\nRepeated combinations:")
print(repeated_sales_keys)

Repeated Order_ID + SKU_ID + Store_ID combinations: 138

Repeated combinations:
Order_ID    SKU_ID    Store_ID
ORD1065788  SKU00821  AHM-04      2
ORD1008910  SKU00298  MUM-03      2
ORD1001476  SKU00913  JAI-02      2
ORD1059174  SKU00874  MUM-02      2
ORD1073224  SKU00743  BEN-02      2
                                 ..
ORD1060630  SKU00145  PUN-01      2
ORD1055713  SKU00753  JAI-05      2
ORD1025084  SKU01082  BEN-03      2
ORD1005017  SKU00422  AHM-05      2
ORD1014434  SKU00141  HYD-02      2
Name: count, Length: 138, dtype: int64


### Cleaning action 14 result:
#### Order_ID + SKU_ID + Store_ID were not sufficient to uniquely identify sales records. Therefore, a surrogate Sales_ID will be used as the primary key when creating the SQL table.

## Cleaning actions summary table

In [116]:
data_cleaning_actions = pd.DataFrame([
    [1, "Exact Duplicate Removal", "Exact duplicate rows were identified across all seven datasets.", "Removed exact duplicate rows using drop_duplicates().", "Sales: 1,217; Product_Master: 16; Inventory_Daily: 424; Returns: 84; Vendor_Supply: 365; Promotions: 8; Customer_Master: 264. Total removed: 2,378."],
    [2, "Sales SKU Mapping", "Sales SKU_ID values were checked against Product_Master and 2,011 sales rows remained unmatched after standardization.", "Retained unmatched Sales records and added SKU_Mapping_Status rather than deleting valid sales transactions.", "78,372 Sales rows mapped; 2,011 remained unmapped; 1,580 unique unmatched SKU_IDs."],
    [3, "Negative Quantity", "326 Sales records contained Quantity < 0.", "Negative quantities were retained and flagged using Negative_Quantity_Flag. Values were not converted to positive quantities.", "326 records flagged; 15 had corresponding Returns records and 311 had no matching Returns record."],
    [4, "Promotion ID Mismatch", "Non-null Promo_ID values were compared against the Promotions master. Six invalid promotion IDs were identified.", "Original Promo_ID was preserved in Original_Promo_ID. Promotion mapping status was created to distinguish matched promotions, genuine no-promotion records, and invalid/missing promotion references.", "3,145 Sales records contained invalid Promo_ID references; 50,685 had no Promo_ID; 26,553 had matched promotions."],
    [5, "Returns Without Sales Mapping", "150 Returns records contained Order_ID values not found in Sales.", "Records were retained and classified using Sales_Mapping_Status rather than deleted.", "150 unmatched Returns records retained for audit and return-loss analysis."],
    [6, "Vendor Date Cleaning", "Vendor_Supply contained invalid/unparseable values in PO_Date, Expected_Delivery_Date and Actual_Delivery_Date.", "All three columns were converted to datetime using errors='coerce'; invalid values became NaT.", "Invalid date values were retained as missing values rather than assigning assumed dates."],
    [6, "Vendor Delivery Delay and SLA", "Vendor delivery timing was evaluated using Actual_Delivery_Date versus Expected_Delivery_Date.", "Delivery delay and SLA_Status were derived for vendor performance analysis.", "8,912 Late; 20,638 On Time; 80 Early; 455 records where delay could not be calculated."],
    [7, "Store_ID Standardization", "Store_ID values contained formatting differences such as capitalization, spaces, hyphens and underscores.", "Original Store_ID was preserved in Original_Store_ID and the working Store_ID was standardized using trimming and uppercase conversion.", "Store identifiers were standardized for consistent cross-table analysis."],
    [8, "Cancelled Orders", "Cancelled transactions remained in the Sales table.", "Cancelled records were retained and identified using Cancellation_Flag while preserving the original Order_Status.", "5,659 cancelled Sales records retained for cancellation and profit-leakage analysis."],
    [9, "Final Date Conversion", "Date columns across the cleaned datasets required consistent datetime types.", "Converted Sales Order_Date, Inventory Date, Customer Signup_Date, Promotion Start_Date and End_Date, and Returns Return_Date to datetime.", "Invalid dates were converted to NaT; no replacement dates were inferred."],
    [10, "SKU_ID Standardization Across Tables", "SKU_ID formatting was checked across Sales, Product_Master, Inventory_Daily, Returns and Vendor_Supply.", "Standardized SKU_ID using strip() and upper() while preserving meaningful characters such as hyphens and underscores.", "SKU_ID formatting was made consistent across related datasets."],
    [10, "Product Master SKU Validation", "Product_Master SKU_ID uniqueness was checked after standardization.", "Confirmed Product_Master SKU_ID values are unique and used Product_Master as the reference SKU list.", "Product_Master contains 1,202 unique SKU_IDs."],
    [10, "Inventory SKU Mapping", "Inventory SKU_ID values were compared against Product_Master.", "Unmatched Inventory records were retained and classified using SKU_Mapping_Status.", "1,041 Inventory rows and 754 unique SKU_IDs were unmatched."],
    [10, "Returns SKU Mapping", "Returns SKU_ID values were compared against Product_Master.", "Unmatched Return records were retained and classified using SKU_Mapping_Status.", "90 Returns rows had unmatched SKU_IDs."],
    [10, "Vendor SKU Mapping", "Vendor_Supply SKU_ID values were compared against Product_Master.", "Original SKU_ID was preserved in SKU_ID_Original and records were classified using SKU_Mapping_Status.", "Unmatched Vendor SKU records were retained for audit and downstream filtering."],
    [11, "Null Value Analysis", "Missing values were profiled across all datasets and columns.", "Missing values were retained where no reliable replacement could be established.", "NULLs were not blindly filled or deleted because missing values and zero values have different meanings."],
    [11, "Null Treatment – Sales Financial Fields", "Selling_Price, Discount_Amount and Payment_Mode contained missing values.", "Missing values were retained as NULL rather than imputed.", "Financially incomplete records remain identifiable for downstream filtering."],
    [11, "Null Treatment – Inventory", "Closing_Stock and Warehouse_Stock contained missing values.", "Missing stock values were retained as NULL and were not converted to zero.", "Missing stock is distinguished from confirmed zero stock for stockout analysis."],
    [11, "Null Treatment – Product/Vendor/Promotion/Customer/Returns", "Several master and transaction attributes contained missing values.", "Missing values were retained because reliable replacement values were unavailable.", "SQL analysis will use appropriate NULL/IS NOT NULL handling."],
    [12, "Outlier Detection", "Potential numerical outliers were identified using the IQR method across numerical columns.", "Potential outliers were retained because an outlier is not automatically a data error and may represent genuine business activity.", "Outliers remain available for profit-leakage, discount, return, vendor and demand analysis."],
    [12, "Promotion PR0200 Duplicate", "PR0200 appeared twice with the same promotion information, but one record had a missing Promo_Budget.", "Removed the incomplete PR0200 record and retained the complete record.", "One incomplete duplicate removed."],
    [12, "Promotion PROMO_XXXX Ambiguity", "PROMO_XXXX appeared in multiple records with different promotion details.", "Records were retained because they contained distinct information; Promo_ID_Status was used to identify the ambiguous identifier.", "PROMO_XXXX records retained and classified as Ambiguous."],
    [11, "Returns Duplicate Investigation", "Repeated Return_ID records were investigated to distinguish incomplete duplicates from genuine conflicts.", "Incomplete duplicate records were removed where one record contained missing information and the complete corresponding record existed.", "4 incomplete Return duplicate records removed; remaining genuine/conflicting records retained."],
    [12, "Vendor Incomplete Duplicate Investigation", "Repeated Vendor records were investigated using the procurement business key.", "Incomplete duplicate records containing one missing value were removed while complete corresponding records were retained.", "15 incomplete Vendor duplicate records removed."],
    [12, "Vendor Impossible Date Investigation", "Eight Vendor PO_ID groups contained conflicting/impossible date combinations.", "Problematic records were retained and flagged as ImpossibleDateCombo rather than deleted, allowing them to be excluded from valid vendor analysis while preserving audit history.", "Impossible-date records retained for audit and excluded from appropriate vendor analysis."],
    [13, "Customer Incomplete Duplicate Investigation", "Repeated Customer_ID records were investigated to distinguish incomplete duplicates from genuine conflicting records.", "Incomplete duplicate rows containing missing values were removed while complete records were retained.", "17 incomplete Customer duplicate rows removed."],
    [13, "Customer Conflicting Records", "Some repeated Customer_ID records contained different non-missing Signup_Date values.", "Conflicting records were retained because the correct source value could not be determined.", "3 conflicting Customer_ID groups retained for audit/investigation."],
    [14, "Sales Business-Key Investigation", "Order_ID + SKU_ID + Store_ID was tested for uniqueness.", "The combination was found insufficient to uniquely identify every Sales record.", "A surrogate Sales_ID will be used as the SQL primary key."],
    [14, "SQL Primary-Key Investigation", "Business identifiers were investigated across Sales, Returns, Inventory, Vendor_Supply, Promotions and Customer_Master.", "Repeated business identifiers were distinguished from exact duplicates before SQL schema design.", "Primary-key decisions will be based on actual data grain rather than assuming every ID is unique."]
], columns=["Cleaning_Action", "Area", "Finding", "Action_Taken", "Result"])
display(data_cleaning_actions)

,Cleaning_Action,Area,Finding,Action_Taken,Result
0,1,Exact Duplicate Removal,Exact duplicate rows were identified across al...,Removed exact duplicate rows using drop_duplic...,"Sales: 1,217; Product_Master: 16; Inventory_Da..."
1,2,Sales SKU Mapping,Sales SKU_ID values were checked against Produ...,Retained unmatched Sales records and added SKU...,"78,372 Sales rows mapped; 2,011 remained unmap..."
2,3,Negative Quantity,326 Sales records contained Quantity < 0.,Negative quantities were retained and flagged ...,326 records flagged; 15 had corresponding Retu...
3,4,Promotion ID Mismatch,Non-null Promo_ID values were compared against...,Original Promo_ID was preserved in Original_Pr...,"3,145 Sales records contained invalid Promo_ID..."
4,5,Returns Without Sales Mapping,150 Returns records contained Order_ID values ...,Records were retained and classified using Sal...,150 unmatched Returns records retained for aud...
5,6,Vendor Date Cleaning,Vendor_Supply contained invalid/unparseable va...,All three columns were converted to datetime u...,Invalid date values were retained as missing v...
6,6,Vendor Delivery Delay and SLA,Vendor delivery timing was evaluated using Act...,Delivery delay and SLA_Status were derived for...,"8,912 Late; 20,638 On Time; 80 Early; 455 reco..."
7,7,Store_ID Standardization,Store_ID values contained formatting differenc...,Original Store_ID was preserved in Original_St...,Store identifiers were standardized for consis...
8,8,Cancelled Orders,Cancelled transactions remained in the Sales t...,Cancelled records were retained and identified...,"5,659 cancelled Sales records retained for can..."
9,9,Final Date Conversion,Date columns across the cleaned datasets requi...,"Converted Sales Order_Date, Inventory Date, Cu...",Invalid dates were converted to NaT; no replac...


# Automated Data Quality Report

This report summarizes the data profiling, cleaning, validation, and standardization performed on the retail chain dataset.

The raw source data was not modified. Cleaning was performed on separate copies of the original datasets.

The report contains:
- Final data quality summary table i.e, cleaning_actions
- Missing-value analysis i.e, null_report
- duplicate_report
- outlier_report

Automated Data Quality Report generated using Python, containing programmatically generated null, duplicate and outlier validation reports, together with documented cleaning actions.

In [117]:

dataset_for_quality_report = ({"Sales": sales_clean, 
                       "Product_Master": product_clean, 
                       "Inventory_Daily": inventory_clean, 
                       "Returns": returns_clean, 
                       "Vendor_Supply": vendor_clean, 
                       "Promotions": promotions_clean, 
                       "Customer_Master": customer_clean})
dataset_for_quality_report

{'Sales':          Order_ID Order_Date Store_ID Customer_ID     SKU_ID  Quantity  \
 0      ORD1000000 2025-06-01   HYD-03  CUST007684   SKU00565         4   
 1      ORD1000001 2025-03-14   HYD-03  CUST004865   SKU01082         6   
 2      ORD1000002 2025-06-05   BEN-05  CUST006245   SKU00901         2   
 3      ORD1000003 2025-06-23   JAI-02  CUST009196   SKU00524         2   
 4      ORD1000004 2025-03-01   PUN-01  CUST000981   SKU00215         2   
 ...           ...        ...      ...         ...        ...       ...   
 81574  ORD1061636 2025-03-03     PUN4  CUST008513   SKU00498         3   
 81580  ORD1044331 2025-05-03   BEN-05  CUST009277  SKU-19228         3   
 81586  ORD1069400 2025-05-23    KOC04  CUST014291   SKU01047         7   
 81587  ORD1071830 2025-02-07     MUM5  CUST013106   SKU00413         7   
 81599  ORD1017737 2025-06-09   BEN-01  CUST012250   SKU00290         1   
 
        Selling_Price  Discount_Amount Promo_ID  Delivery_Type Payment_Mode  \
 0        

In [118]:
null_report = (pd.concat([pd.DataFrame({
    "Table": name, "Column": df.columns, 
    "Total_Rows": len(df),
    "Null_Count": df.isna().sum().values, 
    "Null_Percentage": (df.isna().sum().values / len(df) * 100).round(2)}) 
                         for name, df in dataset_for_quality_report.items()], ignore_index=True))
null_report

,Table,Column,Total_Rows,Null_Count,Null_Percentage
0,Sales,Order_ID,80383,0,0.00
1,Sales,Order_Date,80383,612,0.76
2,Sales,Store_ID,80383,0,0.00
3,Sales,Customer_ID,80383,0,0.00
4,Sales,SKU_ID,80383,0,0.00
...,...,...,...,...,...
66,Promotions,Promo_ID_Status,201,0,0.00
67,Customer_Master,Customer_ID,15019,0,0.00
68,Customer_Master,City,15019,139,0.93
69,Customer_Master,Signup_Date,15019,150,1.00


In [119]:
duplicate_report = pd.DataFrame([{
    "Table": name, 
    "Total_Rows": len(df), 
    "Duplicate_Rows": df.duplicated().sum(), 
    "Duplicate_Percentage": round(df.duplicated().sum() / len(df) * 100, 2)}
                                 for name, df in dataset_for_quality_report.items()])
duplicate_report

,Table,Total_Rows,Duplicate_Rows,Duplicate_Percentage
0,Sales,80383,0,0.0
1,Product_Master,1202,0,0.0
2,Inventory_Daily,50076,0,0.0
3,Returns,4617,0,0.0
4,Vendor_Supply,30070,0,0.0
5,Promotions,201,0,0.0
6,Customer_Master,15019,0,0.0


In [120]:
outlier_report = (pd.DataFrame([
    {"Table": name,
     "Column": col,
     "Outlier_Count": ((df[col] < df[col].quantile(0.25) -1.5 * (df[col].quantile(0.75) - df[col].quantile(0.25)))|
                       (df[col] > df[col].quantile(0.75) +1.5 * (df[col].quantile(0.75) - df[col].quantile(0.25)))).sum()}
    for name, df in dataset_for_quality_report.items()
    for col in df.select_dtypes(include="number").columns]))
outlier_report

,Table,Column,Outlier_Count
0,Sales,Quantity,622
1,Sales,Selling_Price,3390
2,Sales,Discount_Amount,5164
3,Product_Master,Tax_Percent,0
4,Product_Master,Unit_Cost,38
5,Product_Master,MRP,55
6,Inventory_Daily,Reorder_Level,0
7,Inventory_Daily,Opening_Stock,0
8,Inventory_Daily,Closing_Stock,0
9,Inventory_Daily,Warehouse_Stock,0


## Export Cleaned Excel Dataframes and DataQuality Report

In [121]:
# pd.ExcelWriter(...) creates one Excel workbook.
# engine="openpyxl" is a library that works with excel files
# index=False means pandas will not add the DataFrame's row-number index as an extra column.
with pd.ExcelWriter("Retail_Cleaned_Data.xlsx",engine="openpyxl") as writer:
    for name,df in dataset_for_quality_report.items():
        df.to_excel(writer,sheet_name=name,index=False)


In [122]:
# export Data_Quality_Report
with pd.ExcelWriter("Retail_Data_Quality_Report.xlsx", engine="openpyxl") as writer:
    data_cleaning_actions.to_excel(writer, sheet_name="Cleaning_Actions", index=False)
    null_report.to_excel(writer, sheet_name="Null_Report", index=False)
    duplicate_report.to_excel(writer, sheet_name="Duplicate_Report", index=False)
    outlier_report.to_excel(writer, sheet_name="Outlier_Report", index=False)

In [123]:
print("Cleaned Data Workbook:", pd.ExcelFile("Retail_Cleaned_Data.xlsx").sheet_names)
print("Data Quality Report:", pd.ExcelFile("Retail_Data_Quality_Report.xlsx").sheet_names)

Cleaned Data Workbook: ['Sales', 'Product_Master', 'Inventory_Daily', 'Returns', 'Vendor_Supply', 'Promotions', 'Customer_Master']
Data Quality Report: ['Cleaning_Actions', 'Null_Report', 'Duplicate_Report', 'Outlier_Report']


In [124]:
import os
print(os.path.abspath("Retail_Cleaned_Data.xlsx"))
print(os.path.abspath("Retail_Data_Quality_Report.xlsx"))

C:\Users\manit\Retail Chain Profit Leakage & Demand Optimization\Retail_Cleaned_Data.xlsx
C:\Users\manit\Retail Chain Profit Leakage & Demand Optimization\Retail_Data_Quality_Report.xlsx


In [125]:
%%writefile data_cleaning.py
"""
Retail Chain Profit Leakage & Demand Optimization
Data Cleaning Script

This script reproduces the finalized cleaning workflow from
cleaned_data(4).ipynb up to Cleaning Action 14.

Outputs:
    1. Retail_Cleaned_Data.xlsx
    2. Retail_Data_Quality_Report.xlsx
"""

import pandas as pd
from pathlib import Path
# ============================================================
# 1. FILE CONFIGURATION
# ============================================================

RAW_FILE = Path(r"C:\Users\manit\OneDrive\Desktop\PROJECT_1\Retail_Profit_Leakage_Dataset.xlsx")
CLEANED_OUTPUT = Path("Retail_Cleaned_Data.xlsx")
QUALITY_OUTPUT = Path("Retail_Data_Quality_Report.xlsx")
# ============================================================
# 2. LOAD RAW DATA
# ============================================================

def load_data():
    excel_data = pd.ExcelFile(RAW_FILE)

    sales = pd.read_excel(excel_data, sheet_name="Sales_Transactions")
    product = pd.read_excel(excel_data, sheet_name="Product_Master")
    inventory = pd.read_excel(excel_data, sheet_name="Inventory_Daily")
    returns = pd.read_excel(excel_data, sheet_name="Returns")
    vendor = pd.read_excel(excel_data, sheet_name="Vendor_Supply")
    promotions = pd.read_excel(excel_data, sheet_name="Promotions")
    customer = pd.read_excel(excel_data, sheet_name="Customer_Master")

    return {
        "Sales": sales,
        "Product_Master": product,
        "Inventory_Daily": inventory,
        "Returns": returns,
        "Vendor_Supply": vendor,
        "Promotions": promotions,
        "Customer_Master": customer
    }

# ============================================================
# 3. CREATE CLEAN COPIES
# ============================================================
def create_clean_copies(tables):
    return {name: df.copy()
        for name, df in tables.items()}

# ============================================================
# 4. REMOVE EXACT DUPLICATES
# ============================================================

def remove_exact_duplicates(tables):
    for name in tables:tables[name] = tables[name].drop_duplicates()
    return tables

# ============================================================
# 5. SALES SKU MAPPING
# ============================================================
def process_sales_sku_mapping(tables):
    sales = tables["Sales"]
    product = tables["Product_Master"]
    sales["Original_SKU_ID"] = sales["SKU_ID"].copy()
    sales_sku_standard = (sales["SKU_ID"].astype(str).str.strip().str.upper())
    product_sku_standard = (product["SKU_ID"].astype(str).str.strip().str.upper())
    matched_sku = sales_sku_standard.isin(product_sku_standard)
    sales["SKU_Mapping_Status"] = "Matched"
    sales.loc[~matched_sku, "SKU_Mapping_Status"] = "Missing"
    tables["Sales"] = sales
    return tables

# ============================================================
# 6. NEGATIVE QUANTITY
# ============================================================
def process_negative_quantity(tables):
    sales = tables["Sales"]
    sales["Negative_Quantity_Flag"] = "No"
    sales.loc[sales["Quantity"] < 0,"Negative_Quantity_Flag"] = "Yes"
    tables["Sales"] = sales
    return tables

# ============================================================
# 7. PROMO ID MISMATCH
# ============================================================
def process_promotion_mapping(tables):
    sales = tables["Sales"]
    promotions = tables["Promotions"]
    sales["Original_Promo_ID"] = sales["Promo_ID"].copy()
    invalid_promo_ids = sales.loc[sales["Promo_ID"].notna()& ~sales["Promo_ID"].isin(promotions["Promo_ID"]),"Promo_ID"].unique()
    sales.loc[sales["Promo_ID"].isin(invalid_promo_ids),"Promo_ID"] = pd.NA
    sales["Promo_Mapping_Status"] = "Matched"
    sales.loc[sales["Original_Promo_ID"].isna(),"Promo_Mapping_Status"] = "No Promotion"
    sales.loc[sales["Original_Promo_ID"].isin(invalid_promo_ids),"Promo_Mapping_Status"] = "Missing"
    tables["Sales"] = sales
    return tables

# ============================================================
# 8. RETURNS WITHOUT SALES MAPPING
# ============================================================
def process_return_sales_mapping(tables):
    sales = tables["Sales"]
    returns = tables["Returns"]
    returns["Sales_Mapping_Status"] = "Matched"
    returns.loc[~returns["Order_ID"].isin(sales["Order_ID"]),"Sales_Mapping_Status"]= "Missing"
    tables["Returns"] = returns
    return tables

# ============================================================
# 9. VENDOR DATE CLEANING
# ============================================================
def process_vendor_dates(tables):
    vendor = tables["Vendor_Supply"]
    for column in [
        "PO_Date",
        "Expected_Delivery_Date",
        "Actual_Delivery_Date"]:
        vendor[column] = pd.to_datetime(vendor[column],errors="coerce")
    vendor["Delay_Days"] = (vendor["Actual_Delivery_Date"]- vendor["Expected_Delivery_Date"]).dt.days
    vendor["SLA_Status"] = "On Time"
    vendor.loc[vendor["Delay_Days"] > 0,"SLA_Status"] = "Late"
    vendor.loc[vendor["Delay_Days"] < 0,"SLA_Status" ] = "Early"
    vendor.loc[vendor["Delay_Days"].isna(),"SLA_Status"] = "Missing"
    tables["Vendor_Supply"] = vendor
    return tables
# ============================================================
# 10. STORE ID STANDARDIZATION
# ============================================================
def process_store_id(tables):
    sales = tables["Sales"]
    sales["Original_Store_ID"] = sales["Store_ID"].copy()
    sales["Store_ID"] = (sales["Store_ID"].astype(str).str.strip().str.upper() )
    tables["Sales"] = sales
    return tables
# ============================================================
# 11. CANCELLED ORDERS
# ============================================================
def process_cancelled_orders(tables):
    sales = tables["Sales"]
    sales["Cancellation_Flag"] = "No"
    sales.loc[sales["Order_Status"] == "Cancelled","Cancellation_Flag" ] = "Yes"
    tables["Sales"] = sales
    return tables

# ============================================================
# 12. DATE CONVERSION
# ============================================================
def convert_dates(tables):
    tables["Sales"]["Order_Date"] = pd.to_datetime(tables["Sales"]["Order_Date"],errors="coerce")
    tables["Inventory_Daily"]["Date"] = pd.to_datetime(tables["Inventory_Daily"]["Date"],errors="coerce"  )
    tables["Returns"]["Return_Date"] = pd.to_datetime(tables["Returns"]["Return_Date"],errors="coerce"  )
    tables["Promotions"]["Start_Date"] = pd.to_datetime(tables["Promotions"]["Start_Date"],errors="coerce"  )
    tables["Promotions"]["End_Date"] = pd.to_datetime(tables["Promotions"]["End_Date"],errors="coerce"  )
    tables["Customer_Master"]["Signup_Date"] = pd.to_datetime(tables["Customer_Master"]["Signup_Date"],errors="coerce")
    return tables
# ============================================================
# 13. SKU STANDARDIZATION ACROSS TABLES
# ============================================================
def standardize_sku_ids(tables):
    inventory = tables["Inventory_Daily"]
    returns = tables["Returns"]
    vendor = tables["Vendor_Supply"]
    inventory["SKU_ID_Original"] = inventory["SKU_ID"].copy()
    returns["SKU_ID_Original"] = returns["SKU_ID"].copy()
    vendor["SKU_ID_Original"] = vendor["SKU_ID"].copy()
    for df in [tables["Sales"],tables["Product_Master"],inventory,returns,vendor]:
        df["SKU_ID"] = (df["SKU_ID"].astype("string").str.strip().str.upper())
    return tables

# ============================================================
# 14. SKU MAPPING STATUS FOR OTHER TABLES
# ============================================================
def process_other_sku_mappings(tables):
    product_skus = set(tables["Product_Master"]["SKU_ID"].astype(str).str.strip().str.upper())
    for name in ["Inventory_Daily","Returns","Vendor_Supply"]:
        df = tables[name]
        df["SKU_Mapping_Status"] = "Mapped"
        df.loc[~df["SKU_ID"].isin(product_skus),"SKU_Mapping_Status"] = "Unmapped"
        tables[name] = df
    return tables
# ============================================================
# 15. RETURNS INCOMPLETE DUPLICATES
# ============================================================
def process_return_duplicates(tables):
    returns = tables["Returns"]
    duplicate_groups = returns[returns.duplicated(subset=["Return_ID"],keep=False)]
    incomplete_duplicates = duplicate_groups[duplicate_groups.isna().any(axis=1) ]
    returns = returns.drop(incomplete_duplicates.index)
    tables["Returns"] = returns
    return tables

# ============================================================
# 16. PROMOTION PR0200
# ============================================================
def process_promotion_duplicate(tables):
    promotions = tables["Promotions"]
    promotions = promotions.drop(promotions[(promotions["Promo_ID"] == "PR0200")& promotions["Promo_Budget"].isna()].index)
    tables["Promotions"] = promotions
    return tables

# ============================================================
# 17. PROMO_XXXX AMBIGUITY
# ============================================================
def flag_ambiguous_promotion(tables):
    promotions = tables["Promotions"]
    promotions["Promo_ID_Status"] = "Valid"
    promotions.loc[promotions["Promo_ID"] == "PROMO_XXXX","Promo_ID_Status"] = "Ambiguous"
    tables["Promotions"] = promotions
    return tables

# ============================================================
# 18. VENDOR INCOMPLETE DUPLICATES
# ============================================================
def process_vendor_duplicates(tables):
    vendor = tables["Vendor_Supply"]
    duplicate_groups = vendor[vendor.duplicated(subset=["PO_ID"],keep=False)]
    incomplete_duplicates = duplicate_groups[duplicate_groups.isna().any(axis=1)]
    vendor = vendor.drop(incomplete_duplicates.index)
    tables["Vendor_Supply"] = vendor
    return tables
# ===========================================================
# 19. VENDOR IMPOSSIBLE DATE COMBINATIONS
# ============================================================
def flag_vendor_date_quality(tables):
    vendor = tables["Vendor_Supply"]
    vendor["Data_Quality_Flag"] = (
        (vendor["Actual_Delivery_Date"]< vendor["PO_Date"])|
        (vendor["Actual_Delivery_Date"]< vendor["Expected_Delivery_Date"])).map({True: "ImpossibleDateCombo",False: "Normal"})
    tables["Vendor_Supply"] = vendor
    return tables

# ============================================================
# 20. CUSTOMER INCOMPLETE DUPLICATES
# ============================================================
def process_customer_duplicates(tables):
    customer = tables["Customer_Master"]
    duplicate_groups = customer[customer.duplicated(subset=["Customer_ID"],keep=False)]
    incomplete_duplicates = duplicate_groups[duplicate_groups.isna().any(axis=1)]
    customer = customer.drop(incomplete_duplicates.index)
    tables["Customer_Master"] = customer
    return tables
# ============================================================
# 21. CUSTOMER CONFLICTS
# ============================================================
def flag_customer_conflicts(tables):
    customer = tables["Customer_Master"]
    customer["Customer_Record_Status"] = "Valid"
    conflicting_ids = ["CUST009738","CUST006624","CUST008235"]
    customer.loc[customer["Customer_ID"].isin(conflicting_ids),"Customer_Record_Status"] = "Conflict"
    tables["Customer_Master"] = customer
    return tables

# ============================================================
# 22. CLEANING ACTIONS REPORT
# ============================================================
def build_cleaning_actions():
    return pd.DataFrame(
        [
            [1, "Exact Duplicates", "Exact duplicate records found in all seven tables.", "Removed exact duplicate rows.", "All seven cleaned tables validated with zero exact duplicates."],
            [2, "Sales SKU Mapping", "Sales contained unmatched SKU_ID values.", "Retained records and added SKU_Mapping_Status.", "Unmatched Sales records retained and flagged."],
            [3, "Negative Quantity", "Negative Quantity records were identified.", "Retained values and added Negative_Quantity_Flag.", "Negative quantities were not converted or deleted."],
            [4, "Promo ID Mismatch", "Invalid Promo_ID values were identified.", "Preserved Original_Promo_ID and created Promo_Mapping_Status.", "Invalid promotions flagged as Missing."],
            [5, "Returns Mapping", "Returns existed without matching Sales Order_ID.", "Created Sales_Mapping_Status.", "Unmatched Returns retained and flagged."],
            [6, "Vendor Dates", "Invalid Vendor date values were identified.", "Converted dates using errors='coerce'.", "Invalid dates became missing values."],
            [6, "Vendor SLA", "Vendor delivery timing needed classification.", "Created Delay_Days and SLA_Status.", "Late, On Time, Early and Missing statuses created."],
            [7, "Store_ID", "Store_ID formatting was inconsistent.", "Preserved Original_Store_ID and standardized Store_ID.", "Store_ID standardized using strip and upper."],
            [8, "Cancelled Orders", "Cancelled records appeared in Sales.", "Created Cancellation_Flag.", "Cancelled records retained."],
            [9, "Date Conversion", "Date fields required consistent datetime types.", "Converted date columns to datetime.", "Date columns standardized."],
            [10, "SKU_ID Standardization", "SKU_ID formatting varied across tables.", "Preserved original values where required and standardized SKU_ID.", "SKU_ID standardized."],
            [10, "SKU Mapping", "Some SKU differences required mapping investigation.", "Applied confirmed Product_Master mappings only.", "Unresolved mappings retained and flagged."],
            [11, "Returns Duplicates", "Incomplete Return duplicates were identified.", "Removed incomplete duplicate records.", "Complete records retained."],
            [12, "Promotion Duplicate", "PR0200 had an incomplete duplicate.", "Removed the incomplete PR0200 record.", "Complete PR0200 record retained."],
            [12, "Promotion Ambiguity", "PROMO_XXXX had conflicting details.", "Flagged Promo_ID_Status as Ambiguous.", "Ambiguous record retained."],
            [12, "Vendor Duplicates", "Incomplete Vendor duplicates were identified.", "Removed incomplete duplicate records.", "Complete records retained."],
            [12, "Vendor Date Quality", "Impossible Vendor date combinations were identified.", "Created Data_Quality_Flag.", "Impossible combinations retained and flagged."],
            [13, "Customer Duplicates", "Incomplete Customer duplicates were identified.", "Removed incomplete duplicate records.", "Complete records retained."],
            [13, "Customer Conflicts", "Conflicting Customer records remained.", "Flagged conflicting Customer_ID records.", "Conflicting records retained."],
            [14, "Sales Business Key", "Order_ID + SKU_ID + Store_ID was not unique.", "Surrogate Sales_ID will be used in SQL.", "SQL-layer key decision; not a cleaning transformation."]
        ],columns=[
            "Cleaning_Action",
            "Area",
            "Finding",
            "Action_Taken",
            "Result"])
# ============================================================
# 23. NULL REPORT
# ============================================================
def build_null_report(tables):
    return pd.concat([pd.DataFrame({
                    "Table": name,
                    "Column": df.columns,
                    "Total_Rows": len(df),
                    "Null_Count": df.isna().sum().values,
                    "Null_Percentage": (df.isna().sum().values/ len(df)* 100).round(2)})
            for name, df in tables.items()],ignore_index=True)
# ============================================================
# 24. DUPLICATE REPORT
# ============================================================
def build_duplicate_report(tables):
    return pd.DataFrame([{
                "Table": name,
                "Total_Rows": len(df),
                "Duplicate_Rows": df.duplicated().sum(),
                "Duplicate_Percentage": round(df.duplicated().sum()/ len(df)* 100,2)}
            for name, df in tables.items()])

# ============================================================
# 25. OUTLIER REPORT
# ============================================================
def build_outlier_report(tables):
    return pd.DataFrame([{
                "Table": name,
                "Column": col,
                "Outlier_Count": ((df[col]< df[col].quantile(0.25)- 1.5 * (df[col].quantile(0.75)- df[col].quantile(0.25)))
                    |(df[col]> df[col].quantile(0.75)+ 1.5 * (df[col].quantile(0.75)- df[col].quantile(0.25)))).sum()}
            for name, df in tables.items()
            for col in df.select_dtypes(include="number").columns])
# ============================================================
# 26. EXPORT CLEANED DATA
# ============================================================
def export_cleaned_data(tables):
    with pd.ExcelWriter(CLEANED_OUTPUT,engine="openpyxl") as writer:
        for name, df in tables.items():
            df.to_excel(writer,sheet_name=name,index=False)

# ============================================================
# 27. EXPORT DATA QUALITY REPORT
# ============================================================
def export_quality_report(cleaning_actions,null_report,duplicate_report,outlier_report):
    with pd.ExcelWriter(QUALITY_OUTPUT,engine="openpyxl") as writer:
        cleaning_actions.to_excel(writer,sheet_name="Cleaning_Actions",index=False)
        null_report.to_excel(writer,sheet_name="Null_Report",index=False)
        duplicate_report.to_excel(writer,sheet_name="Duplicate_Report",index=False)
        outlier_report.to_excel(writer,sheet_name="Outlier_Report",index=False)

# ============================================================
# 28. MAIN PIPELINE
# ============================================================
def main():
    print("Starting Retail Data Cleaning Pipeline...")
    raw_tables = load_data()
    clean_tables = create_clean_copies(raw_tables)
    clean_tables = remove_exact_duplicates(clean_tables)
    clean_tables = process_sales_sku_mapping(clean_tables)
    clean_tables = process_negative_quantity(clean_tables)
    clean_tables = process_promotion_mapping(clean_tables)
    clean_tables = process_return_sales_mapping(clean_tables)
    clean_tables = process_vendor_dates(clean_tables)
    clean_tables = process_store_id(clean_tables)
    clean_tables = process_cancelled_orders(clean_tables)
    clean_tables = convert_dates(clean_tables)
    clean_tables = standardize_sku_ids(clean_tables)
    clean_tables = process_other_sku_mappings(clean_tables)
    clean_tables = process_return_duplicates(clean_tables)
    clean_tables = process_promotion_duplicate(clean_tables)
    clean_tables = flag_ambiguous_promotion(clean_tables)
    clean_tables = process_vendor_duplicates(clean_tables)
    clean_tables = flag_vendor_date_quality(clean_tables)
    clean_tables = process_customer_duplicates(clean_tables)
    clean_tables = flag_customer_conflicts(clean_tables)
    
    cleaning_actions = build_cleaning_actions()

    null_report = build_null_report(clean_tables)
    duplicate_report = build_duplicate_report(clean_tables)
    outlier_report = build_outlier_report(clean_tables)

    export_cleaned_data(clean_tables)

    export_quality_report(
        cleaning_actions,
        null_report,
        duplicate_report,
        outlier_report)
    print()
    print("Cleaning completed successfully.")
    print("Created:", CLEANED_OUTPUT)
    print("Created:", QUALITY_OUTPUT)
    print()
    print("Duplicate validation:")
    print(duplicate_report)
# ============================================================
# 29. RUN SCRIPT
# ============================================================
if __name__ == "__main__":
    main()

Overwriting data_cleaning.py


In [126]:
%run data_cleaning.py

Starting Retail Data Cleaning Pipeline...

Cleaning completed successfully.
Created: Retail_Cleaned_Data.xlsx
Created: Retail_Data_Quality_Report.xlsx

Duplicate validation:
             Table  Total_Rows  Duplicate_Rows  Duplicate_Percentage
0            Sales       80383               0                   0.0
1   Product_Master        1202               0                   0.0
2  Inventory_Daily       50076               0                   0.0
3          Returns        4613               0                   0.0
4    Vendor_Supply       30067               0                   0.0
5       Promotions         201               0                   0.0
6  Customer_Master       15019               0                   0.0


In [127]:
from pathlib import Path
print(Path("Retail_Cleaned_Data.xlsx").exists())
print(Path("Retail_Data_Quality_Report.xlsx").exists())

True
True


In [128]:
import pandas as pd
print(pd.ExcelFile("Retail_Cleaned_Data.xlsx").sheet_names)

['Sales', 'Product_Master', 'Inventory_Daily', 'Returns', 'Vendor_Supply', 'Promotions', 'Customer_Master']


In [129]:
import pandas as pd
print(pd.ExcelFile("Retail_Data_Quality_Report.xlsx").sheet_names)

['Cleaning_Actions', 'Null_Report', 'Duplicate_Report', 'Outlier_Report']


If the company gives you another file with the same seven tables and same column structure, you can potentially run the pipeline again with minimal changes to the input filename.
The report can automatically recalculate:

- missing values
- percentages
- duplicates
- affected records
- cleaned row counts
- validation results

# Customer Segmentation

## calculating the four required customer metrics.

four purchase-behavior metrics:

- Total Spend → how much the customer spent
- Average Order Value (AOV) → average amount spent per order
- Order Frequency → how many orders the customer made
- Return Frequency → how many returns the customer made

In [191]:
# Keep valid completed purchases for customer segmentation
# without changing your original cleaned sales data.
# valid_sales= rows from sales_clean that satisfy all the conditions inside the square brackets.
valid_sales = sales_clean.loc[(sales_clean["Order_Status"] == "Delivered") &
    (sales_clean["Negative_Quantity_Flag"] == "No") &
    (sales_clean["Quantity"] > 0) &
    (sales_clean["Customer_ID"].notna()) &
    (sales_clean["Selling_Price"].notna()) &
    (sales_clean["Discount_Amount"].notna())& 
    (sales_clean["Order_ID"].notna())].copy()

# Calculate the net amount paid for each sales record
# This is sales value after discounts, not profit(bcoz we've not included COGS and Returns).
valid_sales["Net_Sales"] = (valid_sales["Quantity"] * valid_sales["Selling_Price"]- valid_sales["Discount_Amount"])

In [203]:
# Create customer-level purchase metrics
# turns sales-level records into one row per customer.
# agg means aggregate. It allows us to calculate several summaries for each customer in one operation.
# After grouping, Customer_ID becomes the grouping index. reset_index() returns it to a normal column 
#so we can work with it and merge it with other DataFrames.
customer_metrics = (valid_sales .groupby("Customer_ID").agg(
        # Adds the customer's Net_Sales values by customer
        Total_Spend=("Net_Sales", "sum"),
        Average_Order_Value=("Net_Sales", "mean"),
        Order_Frequency=("Order_ID", "nunique")).reset_index())

customer_metrics.head(5)

,Customer_ID,Total_Spend,Average_Order_Value,Order_Frequency
0,CUST000001,1210.07,403.356667,3
1,CUST000002,1493.36,165.928889,9
2,CUST000003,1124.25,224.850000,5
3,CUST000004,1001.85,333.950000,3
4,CUST000005,348.77,174.385000,2


In [195]:
# Map each return to the customer who placed the original order
# merge = connects returns to sales using the order ID.
# how="left" preserves all rows from returns_clean, even when no matching order is found.

return_customer = (returns_clean.merge
# selects the order ID and customer ID from Sales.
(sales_clean[["Order_ID", "Customer_ID"]],on="Order_ID",how="left"))

return_customer.head()

,Return_ID,Order_ID,SKU_ID,Return_Date,Return_Reason,Refund_Amount,Return_Status,Sales_Mapping_Status,SKU_ID_Original,SKU_Mapping_Status,Customer_ID
0,RET200000,ORD1079072,SKU00780,2025-05-11,Quality Issue,187.78,Approved,Matched,SKU00780,Mapped,CUST010653
1,RET200001,ORD1032213,SKU00428,2025-06-29,Quality Issue,72.06,Approved,Matched,SKU00428,Mapped,CUST011451
2,RET200002,ORD1004967,SKU00969,2025-07-02,Changed Mind,279.44,Approved,Matched,SKU00969,Mapped,CUST010558
3,RET200003,ORD1042110,SKU00300,2025-04-19,Quality Issue,195.15,Approved,Matched,SKU00300,Mapped,CUST001350
4,RET200004,ORD1005077,SKU00528,2025-01-17,Quality Issue,427.84,Approved,Matched,SKU00528,Mapped,CUST008395


In [199]:
# dropna(subset) removes rows where the customer ID is missing. This is necessary because we cannot calculate a customer-level 
# return count when we do not know which customer made the order.
# "Return_ID" is the name of the column we want to calculate something on. "nunique" is name of the aggregation operation we want pandas to perform
# reset_index makes Customer_ID again a regular column and not as INDEX

return_frequency=(return_customer.dropna(subset=["Customer_ID"]).groupby("Customer_ID").agg(Return_Frequency=("Return_ID", "nunique")).reset_index())

return_frequency.head(5)

,Customer_ID,Return_Frequency
0,CUST000002,1
1,CUST000003,1
2,CUST000004,1
3,CUST000005,1
4,CUST000018,1


In [207]:
## Now we need to bring Return_Frequency into customer_metrics.
# customer_metrics has: "Customer_ID, Total_Spend, Average_Order_Value, Order_Frequency"
# And return_frequency has: "Customer_ID, Return_Frequency"
# Total Spend + Average Order Value + Order Frequency + Return Frequency
# Customers who have no returns should get 0, not a blank value.

customer_metrics = customer_metrics.merge(return_frequency,on="Customer_ID",how="left")

customer_metrics["Return_Frequency"] = (customer_metrics["Return_Frequency"].fillna(0).astype(int))


In [210]:
customer_metrics.head()

,Customer_ID,Total_Spend,Average_Order_Value,Order_Frequency,Return_Frequency
0,CUST000001,1210.07,403.356667,3,0
1,CUST000002,1493.36,165.928889,9,1
2,CUST000003,1124.25,224.850000,5,1
3,CUST000004,1001.85,333.950000,3,1
4,CUST000005,348.77,174.385000,2,1


In [226]:
# Average Order Value tells us how much customers spend per order, on average.

customer_metrics["Average_Order_Value"] = customer_metrics["Total_Spend"] / customer_metrics["Order_Frequency"]

Inspect the metrics before clustering

Before applying K-Means, we should understand the distribution of these four variables

In [225]:
# Check the basic statistics of the four customer segmentation metrics
customer_metrics[["Total_Spend","Average_Order_Value","Order_Frequency","Return_Frequency"]].describe()

,Total_Spend,Average_Order_Value,Order_Frequency,Return_Frequency
count,14901.000000,14901.000000,14901.000000,14901.000000
mean,1523.358958,308.132827,4.933562,0.299443
std,930.542592,145.164182,2.185711,0.545136
min,23.160000,23.160000,1.000000,0.000000
25%,836.720000,211.297500,3.000000,0.000000
50%,1363.610000,282.945000,5.000000,0.000000
75%,2041.320000,372.693333,6.000000,1.000000
max,7649.490000,2005.700000,16.000000,4.000000


- Total Spend: ₹23 → ₹7,649
- Average Order Value: ₹23 → ₹2,006
- Order Frequency: 1 → 16 orders
- Return Frequency: 0 → 4 returns

### Scale the four metrics
Scaling puts all four variables onto a comparable scale while keeping the original values in customer_metrics unchanged.

We'll use StandardScaler, which is a standard preprocessing technique for K-Means.

Imagine you own a retail chain and have 14,901 customers. You want to divide them into groups based on how they shop.
For example:
- Customers who spend a lot(Total Spend).
- Customers who shop frequently(Order_Frequency).
- Customers who return products frequently(Return_Frequency).
- Customers who spend less and shop less often(Average_Order_Value).

This is customer segmentation — dividing customers into groups based on similar behaviour.

In [228]:
# sklearn = Python library containing tools for machine learning.
# preprocessing = contains tools to prepare data before using machine learning.
# StandardScaler is one of those tools. we will use to make our numbers(of very different sizes) comparable.

from sklearn.preprocessing import StandardScaler

# Select the four metrics required for customer segmentation ie,these are the four columns I want to use for grouping customers.”
segmentation_features = ["Total_Spend","Average_Order_Value","Order_Frequency","Return_Frequency"]

# Scale the features so they have comparable influence on clustering
# StandardScaler() creates a new scaler. scaler is the variable where we store that tool.
scaler = StandardScaler()


In [229]:
# segmentation_features contains our four column names, Python selects those four columns from customer_metrics.
# It does not select Customer_ID for the clustering calculations. We are using the 4 numerical features to measure customer behaviour.

# fit — learn the numbers. For example, it learns the typical Total Spend and how much customers spending varies.
# transform — change the scale. It converts the original values into standardized values.

customer_scaled = scaler.fit_transform(customer_metrics[segmentation_features])
print("Customers:", customer_scaled.shape[0])
print("Segmentation features:", customer_scaled.shape[1])

Customers: 14901
Segmentation features: 4


 #### whole process:
- customer_metrics--> Select the four features
- StandardScaler--> Learns each feature's scale and standardizes its values
- customer_scaled--> The scaled numerical data, ready for clustering

#### Decide the number of customer clusters
The Silhouette Score tells us how well-separated the customer groups are.

Higher score → customers within a group are more similar and groups are better separated.
We'll use this only to help choose the cluster count; we won't assign business labels yet.

In [230]:
# cluster contains tools for grouping similar data.
# KMeans is the tool that creates the groups which are similar together.
# silhouette_score= imports a tool that evaluates how well the groups are separated. 
# Think of it as a teacher checking whether students in each group are similar to one another and different from students in other groups.


from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Test the required range of 4 to 6 customer clusters
# n_clusters=k: tells the model how many groups to create. When k=4, it creates four groups.
# random_state=42: helps make the results reproducible for no.of loop times
# n_init=10: the algorithm tries 10 different initial starting arrangements and keeps the best result according to its clustering objective.
for k in [4, 5, 6]:
    kmeans = KMeans(n_clusters=k,random_state=42,n_init=10)
    # fit: the model learns the group centres from your scaled customer data.
    # predict: it assigns every customer to one of the groups.
    cluster_labels = kmeans.fit_predict(customer_scaled)
    score = silhouette_score(customer_scaled, cluster_labels)   
    # {score:.4f} displays the score rounded to four decimal places.
    print(f"Clusters: {k} | Silhouette Score: {score:.4f}")

Clusters: 4 | Silhouette Score: 0.3367
Clusters: 5 | Silhouette Score: 0.3311
Clusters: 6 | Silhouette Score: 0.3327


Among the 4–6 clusters required by the project, 4 clusters has the highest Silhouette Score (0.3376).

So we'll use 4 customer segments.

In [231]:
# Create the final K-Means model with 4 customer clusters
# KMeans(n_clusters=4, ...) creates the final model with four groups.
kmeans_final = KMeans(n_clusters=4,random_state=42,n_init=10)

# Assign each customer to a cluster
# fit_predict(customer_scaled) learns the groups and assigns every customer a cluster label.
# customer_metrics["Cluster"] = ... saves each customer's group assignment in your original customer metrics table.
customer_metrics["Cluster"] = kmeans_final.fit_predict(customer_scaled)

# Check the number of customers in each cluster
customer_metrics["Cluster"].value_counts().sort_index()

Cluster
0    3759
1    6328
2    3082
3    1732
Name: count, dtype: int64

Profile each cluster

We'll calculate the average values of all four metrics for each cluster.

we can give each cluster a business-friendly name based on the actual data.

In [232]:
# Profile the customer segments using the four segmentation metrics
cluster_profile = (customer_metrics.groupby("Cluster")[["Total_Spend","Average_Order_Value","Order_Frequency","Return_Frequency"]].mean().round(2))

cluster_profile

,Total_Spend,Average_Order_Value,Order_Frequency,Return_Frequency
Cluster,,,,
0,2086.09,306.06,6.86,0.00
1,772.31,234.06,3.34,0.09
2,1940.11,306.76,6.35,1.19
3,2304.49,585.70,4.05,0.12


Business-friendly names

Based directly on those profiles, we can use:

- Cluster 0 → High-Value Loyal Customers
- Cluster 1 → Low-Engagement Customers
- Cluster 2 → High-Value High-Return Customers
- Cluster 3 → High-Ticket Customers

In [237]:
# Give each technical cluster a business-friendly name
cluster_names = {
    0: "High-Value Loyal Customers",
    1: "Low-Engagement Customers",
    2: "High-Value High-Return Customers",
    3: "High-Ticket Customers"
}
# customer_metrics["Cluster"] selects the column containing each customer's cluster number.
# .map(cluster_names) looks up the corresponding numnber in your dictionary.
# customer_metrics["Customer_Segment"] = creates a new column and stores the names in it.
customer_metrics["Customer_Segment"] = customer_metrics["Cluster"].map(cluster_names)

# View the final customer segmentation
customer_metrics.head()

,Customer_ID,Total_Spend,Average_Order_Value,Order_Frequency,Return_Frequency,Cluster,Customer_Segment
0,CUST000001,1210.07,403.356667,3,0,1,Low-Engagement Customers
1,CUST000002,1493.36,165.928889,9,1,2,High-Value High-Return Customers
2,CUST000003,1124.25,224.850000,5,1,2,High-Value High-Return Customers
3,CUST000004,1001.85,333.950000,3,1,1,Low-Engagement Customers
4,CUST000005,348.77,174.385000,2,1,1,Low-Engagement Customers


In [240]:
# Final segmentation summary
#Before we move on, let's create a simple summary showing how many customers and what percentage of customers belong to each segment.


# Create a summary of customer segments
# we group by cluster names so one grain= one cluster, then count no.of non-missing customer in each cluster
# reset_index turns Customer_Segment back into a regular column instead of leaving it as the index.
segment_summary = (customer_metrics.groupby("Customer_Segment").agg(Customer_Count=("Customer_ID", "count")).reset_index())

# Calculate the percentage of customers in each segment
# DENOMINATOR has .sum() — adds the customer counts across all four segments.
segment_summary["Customer_Percentage"] = ((segment_summary["Customer_Count"]/ segment_summary["Customer_Count"].sum())* 100).round(2)

segment_summary

,Customer_Segment,Customer_Count,Customer_Percentage
0,High-Ticket Customers,1732,11.62
1,High-Value High-Return Customers,3082,20.68
2,High-Value Loyal Customers,3759,25.23
3,Low-Engagement Customers,6328,42.47


#  Automated Insights Report

This section summarizes the major profit leakage drivers, vendor performance, stockout hotspots, and promotion opportunities identified from the cleaned retail data.

### Step 1 — Top 5 Loss Drivers

In [246]:
# Calculate net profit for each SKU to identify the top loss drivers

# Keep valid delivered sales for profitability analysis. dont change original sales_clean
profit_sales = sales_clean[
    (sales_clean["Order_Status"] == "Delivered") &
    (sales_clean["Negative_Quantity_Flag"] != "Yes") &
    (sales_clean["Quantity"] > 0) &
    (sales_clean["Selling_Price"].notna()) &
    (sales_clean["Discount_Amount"].notna())
].copy()

# Calculate sales after discount
profit_sales["Net_Sales"] = (profit_sales["Quantity"] * profit_sales["Selling_Price"]- profit_sales["Discount_Amount"])

# Add product cost information
# .merge() joins two DataFrames together using a common column.
# inner join bcoz We cannot calculate product cost of that SKU which is not present in product_master so only matching sku's.
profit_sales = profit_sales.merge(product_clean[["SKU_ID", "SKU_Name", "Category", "Unit_Cost"]],on="SKU_ID",how="inner")

# Keep products with a valid unit cost only as some sku's have unit_costs null.
profit_sales = profit_sales[profit_sales["Unit_Cost"].notna() & (profit_sales["Unit_Cost"] > 0)]

# Calculate gross profit after discount
profit_sales["Gross_Profit"] = (profit_sales["Net_Sales"]- profit_sales["Quantity"] * profit_sales["Unit_Cost"])

In [247]:
# Calculate approved refunds by SKU
# Keep approved return records ---> Group them by product SKU ----> Add up their refund amounts.
# pandas normally places the grouping key in the index so .reset_index turns SKU_ID back into a regular column.

approved_returns=(returns_clean[returns_clean["Return_Status"]== "Approved"].groupby("SKU_ID")["Refund_Amount"].sum().reset_index(name="Total_Refund"))
approved_returns.head()

,SKU_ID,Total_Refund
0,SKU-00536,185.58
1,SKU-02642,23.72
2,SKU-05504,825.08
3,SKU-07223,353.50
4,SKU-07488,136.42


In [249]:
# Add refunds to the profit calculation ie, Combine gross profit and approved refunds
# as_index=False keeps the grouping columns as ordinary columns in the result.
# take Gross_Profit column from profit_sales and Add all its values together using "sum". Name the result Gross_Profit.
profit_leakage = (profit_sales
    .groupby(["SKU_ID", "SKU_Name", "Category"], as_index=False).agg(Gross_Profit=("Gross_Profit", "sum"))
    .merge(approved_returns, on="SKU_ID", how="left"))

profit_leakage.head()

,SKU_ID,SKU_Name,Category,Gross_Profit,Total_Refund
0,SKU00001,Snackify Biscuits 500g,Snacks,4361.98,1034.72
1,SKU00002,FreshCo Paneer 250g,Dairy,290.44,596.08
2,SKU00003,DailyDairy Edible Oil 500g,Groceries,334.40,156.28
3,SKU00004,Snackify Butter 1kg,Dairy,6666.53,988.67
4,SKU00005,BabyBloom Detergent 750ml,Home Care,4326.32,2197.99


In [250]:
# SKUs without approved returns have zero refund loss so fill with 0
profit_leakage["Total_Refund"] = profit_leakage["Total_Refund"].fillna(0)

# Final net profit
profit_leakage["Net_Profit"] = (profit_leakage["Gross_Profit"]- profit_leakage["Total_Refund"])

# Select the 5 biggest profit-leaking SKUs
top_5_loss_drivers = (profit_leakage.sort_values("Net_Profit").head(5))

top_5_loss_drivers

,SKU_ID,SKU_Name,Category,Gross_Profit,Total_Refund,Net_Profit
777,SKU00793,UrbanLeaf Rice Combo,Groceries,-1298.16,9989.41,-11287.57
860,SKU00877,UrbanLeaf Ice Cream 1L,Frozen,-2391.55,7544.42,-9935.97
770,SKU00786,HealthHub Diapers Pack,Baby Care,-2776.88,5303.62,-8080.50
750,SKU00766,CleanPro Curd 1kg,Dairy,285.04,7800.39,-7515.35
463,SKU00471,Snackify Rice Combo,Groceries,-2393.37,3636.19,-6029.56


### Step 2 — Worst 3 Vendors
For consistency with the SQL vendor analysis we already completed, we'll evaluate vendors using:

SLA Breach % → percentage of valid POs delivered late
Average Delay Days → average delay among late deliveries
Total PO Count → number of purchase orders

For this first calculation, we'll rank vendors by SLA Breach %, because a higher percentage means the vendor missed the expected delivery date more often.

In [253]:
# Calculate vendor delivery performance

# Keep vendor records into new table where delivery performance can be measured
# df[] square braclets tells pandas to select rows based on conditions.
valid_vendor = vendor_clean[
    vendor_clean["PO_Date"].notna() &
    vendor_clean["Expected_Delivery_Date"].notna() &
    vendor_clean["Actual_Delivery_Date"].notna() &
    (vendor_clean["Data_Quality_Flag"].fillna("Normal") != "ImpossibleDateCombo")
].copy()

# Identify late deliveries create a column named Late and store the results in it.
valid_vendor["Late"] = (valid_vendor["Actual_Delivery_Date"]> valid_vendor["Expected_Delivery_Date"])

valid_vendor.head()

,PO_ID,Vendor_ID,SKU_ID,Store_ID,PO_Date,Expected_Delivery_Date,Actual_Delivery_Date,Supplied_Quantity,Procurement_Cost,Delay_Days,SLA_Status,SKU_ID_Original,SKU_Mapping_Status,Data_Quality_Flag,Late
0,PO300000,VEND0062,SKU00123,DEL-01,2025-03-18,2025-03-24,2025-03-31,57,211.19,7.0,Late,SKU00123,Mapped,Normal,True
1,PO300001,VEND0037,SKU00870,AHM1,2025-03-16,2025-03-21,2025-03-21,52,155.29,0.0,On Time,SKU00870,Mapped,Normal,False
2,PO300002,VEND0007,SKU00030,KOC-02,2025-03-15,2025-03-20,2025-03-22,65,78.13,2.0,Late,SKU00030,Mapped,Normal,True
3,PO300003,VEND0056,SKU01069,MUM-05,2025-04-03,2025-04-10,2025-04-11,53,59.70,1.0,Late,SKU01069,Mapped,Normal,True
4,PO300004,VEND0046,SKU01152,JAI-02,2025-05-04,2025-05-12,2025-05-12,69,51.96,0.0,On Time,SKU01152,Mapped,Normal,False


In [254]:
# Calculate vendor-level performance. We are turning many purchase-order records into one summary row per vendor.
# count all PO_ID count of each Vendor and name new column Total_PO_Count
# take Late columns and sum true values and name new column Late_PO_Count
# tale Delay days columns in valid_vendor and do avg and name new column Average_Delay_Days

vendor_performance = (valid_vendor.groupby("Vendor_ID").agg(Total_PO_Count=("PO_ID", "count"),Late_PO_Count=("Late", "sum"),
        Average_Delay_Days=("Delay_Days", "mean")).reset_index())
vendor_performance.head()

,Vendor_ID,Total_PO_Count,Late_PO_Count,Average_Delay_Days
0,VEND0001,234,85,1.529915
1,VEND0002,246,76,1.882114
2,VEND0003,222,63,1.801802
3,VEND0004,230,70,1.313043
4,VEND0005,257,81,0.817121


In [255]:
# Calculate SLA breach percentage
vendor_performance["SLA_Breach_Percent"] = (vendor_performance["Late_PO_Count"]/ vendor_performance["Total_PO_Count"]* 100).round(2)
vendor_performance.head()

,Vendor_ID,Total_PO_Count,Late_PO_Count,Average_Delay_Days,SLA_Breach_Percent
0,VEND0001,234,85,1.529915,36.32
1,VEND0002,246,76,1.882114,30.89
2,VEND0003,222,63,1.801802,28.38
3,VEND0004,230,70,1.313043,30.43
4,VEND0005,257,81,0.817121,31.52


In [256]:
# Select the 3 vendors with the highest SLA breach percentage
worst_3_vendors = (vendor_performance.sort_values(["SLA_Breach_Percent", "Average_Delay_Days"],ascending=[False, False]).head(3))

worst_3_vendors

,Vendor_ID,Total_PO_Count,Late_PO_Count,Average_Delay_Days,SLA_Breach_Percent
25,VEND0026,226,85,1.858407,37.61
17,VEND0018,239,89,1.246862,37.24
0,VEND0001,234,85,1.529915,36.32


### step 3 Stockout Hotspots

In [338]:
inventory_clean.head()

,Date,Store_ID,SKU_ID,Reorder_Level,Opening_Stock,Closing_Stock,Warehouse_Stock,SKU_ID_Original,SKU_Mapping_Status
0,2025-02-01,MUM-04,SKU01181,10,96,92.0,394.0,SKU01181,Mapped
1,2025-01-25,PUN-01,SKU01164,34,54,52.0,265.0,SKU01164,Mapped
2,2025-01-07,AHM-05,SKU00092,31,40,42.0,324.0,SKU00092,Mapped
3,2025-04-24,KOC-02,SKU00453,13,79,79.0,106.0,SKU00453,Mapped
4,2025-02-05,JAI-05,SKU00507,24,107,103.0,305.0,SKU00507,Mapped


In [339]:
# Identify stockout records with valid dates and identifiers
stockout_inventory = inventory_clean[
    (inventory_clean["Closing_Stock"] == 0) &
    inventory_clean["Date"].notna() &
    inventory_clean["SKU_ID"].notna() &
    inventory_clean["Store_ID"].notna()
].copy()

stockout_inventory.head()

,Date,Store_ID,SKU_ID,Reorder_Level,Opening_Stock,Closing_Stock,Warehouse_Stock,SKU_ID_Original,SKU_Mapping_Status
35,2025-03-12,DEL-05,SKU01035,26,2,0.0,392.0,SKU01035,Mapped
51,2025-03-12,AHM-01,SKU00247,14,1,0.0,156.0,SKU00247,Mapped
64,2025-05-20,DEL-01,SKU00361,13,0,0.0,80.0,SKU00361,Mapped
73,2025-01-18,KOC-04,SKU00046,34,1,0.0,112.0,SKU00046,Mapped
255,2025-03-26,BEN-04,SKU00664,22,0,0.0,322.0,SKU00664,Mapped


In [340]:
# Count stockout days for each product and store
# Put the most frequent stockouts first
stockout_frequency_hotspots = (stockout_inventory.groupby(["SKU_ID", "Store_ID"])
    .agg(Stockout_Days=("Date", "nunique"))
    .reset_index()
    .sort_values("Stockout_Days", ascending=False))

stockout_frequency_hotspots.head()

,SKU_ID,Store_ID,Stockout_Days
701,SKU00665,KOL-05,2
80,SKU00070,KOC-05,2
769,SKU00734,AHM-05,2
1230,SKU01164,DEL-01,2
28,SKU00017,HYD-05,2


In [343]:
sales_clean.head()

,Order_ID,Order_Date,Store_ID,Customer_ID,SKU_ID,Quantity,Selling_Price,Discount_Amount,Promo_ID,Delivery_Type,Payment_Mode,Order_Status,Original_SKU_ID,SKU_Mapping_Status,Negative_Quantity_Flag,Original_Promo_ID,Promo_Mapping_Status,Original_Store_ID,Cancellation_Flag
0,ORD1000000,2025-06-01,HYD-03,CUST007684,SKU00565,4,72.73,19.52,<NA>,Home Delivery,UPI,Delivered,SKU00565,Matched,No,PROMO-9999,Missing,HYD-03,No
1,ORD1000001,2025-03-14,HYD-03,CUST004865,SKU01082,6,273.62,173.52,NaN,Store Pickup,Cash,Delivered,SKU01082,Matched,No,NaN,No Promotion,HYD-03,No
2,ORD1000002,2025-06-05,BEN-05,CUST006245,SKU00901,2,58.95,4.42,NaN,Store Pickup,Card,Delivered,SKU00901,Matched,No,NaN,No Promotion,BEN-05,No
3,ORD1000003,2025-06-23,JAI-02,CUST009196,SKU00524,2,188.08,23.20,NaN,Store Pickup,UPI,Cancelled,SKU00524,Matched,No,NaN,No Promotion,JAI-02,Yes
4,ORD1000004,2025-03-01,PUN-01,CUST000981,SKU00215,2,91.13,16.68,NaN,Home Delivery,Cash,Delivered,SKU00215,Matched,No,NaN,No Promotion,PUN-01,No


In [344]:
# Keep eligible sales for stockout analysis
# Select delivered sales with positive quantities, a valid selling price, and a valid date so we can compare sales against inventory.
stockout_sales = sales_clean[
    (sales_clean["Order_Status"] == "Delivered") &
    (sales_clean["Quantity"] > 0) &
    (sales_clean["Negative_Quantity_Flag"] != "Yes") &
    sales_clean["Selling_Price"].notna() &
    sales_clean["Order_Date"].notna()
].copy()


In [345]:
# Match sales to inventory stockout events
# stockout_demand keeps only stockout records that match eligible sales.
stockout_demand = stockout_inventory.merge(stockout_sales, 
                                           left_on=["SKU_ID", "Store_ID", "Date"], 
                                           right_on=["SKU_ID", "Store_ID", "Order_Date"], how="inner", suffixes=("_Inventory", "_Sales"))
stockout_demand.head()

,Date,Store_ID,SKU_ID,Reorder_Level,Opening_Stock,Closing_Stock,Warehouse_Stock,SKU_ID_Original,SKU_Mapping_Status_Inventory,Order_ID,...,Delivery_Type,Payment_Mode,Order_Status,Original_SKU_ID,SKU_Mapping_Status_Sales,Negative_Quantity_Flag,Original_Promo_ID,Promo_Mapping_Status,Original_Store_ID,Cancellation_Flag
0,2025-02-22,HYD-02,SKU01064,14,4,0.0,202.0,SKU01064,Mapped,ORD1071728,...,Store Pickup,Card,Delivered,SKU01064,Matched,No,NaN,No Promotion,HYD-02,No
1,2025-05-17,KOL-05,SKU01118,30,4,0.0,385.0,SKU01118,Mapped,ORD1020159,...,Home Delivery,UPI,Delivered,SKU01118,Matched,No,PR0020,Matched,KOL-05,No
2,2025-03-29,HYD-02,SKU00202,17,1,0.0,241.0,SKU00202,Mapped,ORD1050623,...,Express,Card,Delivered,SKU00202,Matched,No,NaN,No Promotion,HYD-02,No
3,2025-06-05,PUN-01,SKU00536,21,8,0.0,393.0,SKU00536,Mapped,ORD1006894,...,Home Delivery,Card,Delivered,SKU00536,Matched,No,PR0022,Matched,PUN-01,No
4,2025-01-04,AHM-01,SKU00788,12,2,0.0,254.0,SKU00788,Mapped,ORD1020564,...,Home Delivery,UPI,Delivered,SKU00788,Matched,No,NaN,No Promotion,AHM-01,No


In [346]:
# Estimate sales value associated with recorded demand
# new column called Potential_Sales_Value.
stockout_demand["Potential_Sales_Value"] = (
    stockout_demand["Quantity"] * stockout_demand["Selling_Price"])

In [347]:
# Summarize hotspots by product and store
# as_index=False keeps the group columns as normal columns in the resulting DataFrame

stockout_hotspots = (stockout_demand.groupby(["SKU_ID", "Store_ID"], as_index=False).
    agg(Stockout_Days=("Date", "nunique"), Demand_Quantity=("Quantity", "sum"),
    Potential_Sales_Value=("Potential_Sales_Value", "sum")).
    sort_values(["Stockout_Days", "Potential_Sales_Value"], ascending=[False, False]))

In [348]:
print(stockout_hotspots.to_string(index=False))

  SKU_ID Store_ID  Stockout_Days  Demand_Quantity  Potential_Sales_Value
SKU01064   HYD-02              1                4                 642.32
SKU00202   HYD-02              1                3                 543.39
SKU00536   PUN-01              1                2                 539.86
SKU00081   CHE-02              1                5                 347.40
SKU01118   KOL-05              1                2                 307.22
SKU01104   BEN-01              1                3                 237.66
SKU00788   AHM-01              1                1                 167.03
SKU00098   AHM-02              1                1                 103.61
SKU00241   MUM-02              1                1                  67.87
SKU01093   DEL-04              1                2                  52.30
SKU00018   KOL-01              1                1                  50.92


- **whole process**
 
  inventory_clean(All inventory records)
  
              |
  
  stockout_inventory(Closing stock = 0; valid date, SKU and store)
  
              |
  
  Match with stockout_sales(Same SKU + store + date; inner join)
  
              |
  
  stockout_demand(Only stockout records with matching eligible sales)
  
              |
  
  stockout_hotspots(Stockout days, demand quantity and associated sales value per SKU–STORE)

### Step 4 — Best 5 SKUs to Promote.
We'll identify SKUs that:

- Have Delivered sales.
- Have positive valid quantities.
- Are not negative-quantity records.
- Generate the highest net profit after discounts.
- Show sales volume alongside the profit so we can understand whether the SKU has meaningful demand.

A SKU is considered suitable for promotion when it already generates strong net profit from valid sales.

In [351]:
sales_clean.head(10)

,Order_ID,Order_Date,Store_ID,Customer_ID,SKU_ID,Quantity,Selling_Price,Discount_Amount,Promo_ID,Delivery_Type,Payment_Mode,Order_Status,Original_SKU_ID,SKU_Mapping_Status,Negative_Quantity_Flag,Original_Promo_ID,Promo_Mapping_Status,Original_Store_ID,Cancellation_Flag
0,ORD1000000,2025-06-01,HYD-03,CUST007684,SKU00565,4,72.73,19.52,<NA>,Home Delivery,UPI,Delivered,SKU00565,Matched,No,PROMO-9999,Missing,HYD-03,No
1,ORD1000001,2025-03-14,HYD-03,CUST004865,SKU01082,6,273.62,173.52,NaN,Store Pickup,Cash,Delivered,SKU01082,Matched,No,NaN,No Promotion,HYD-03,No
2,ORD1000002,2025-06-05,BEN-05,CUST006245,SKU00901,2,58.95,4.42,NaN,Store Pickup,Card,Delivered,SKU00901,Matched,No,NaN,No Promotion,BEN-05,No
3,ORD1000003,2025-06-23,JAI-02,CUST009196,SKU00524,2,188.08,23.20,NaN,Store Pickup,UPI,Cancelled,SKU00524,Matched,No,NaN,No Promotion,JAI-02,Yes
4,ORD1000004,2025-03-01,PUN-01,CUST000981,SKU00215,2,91.13,16.68,NaN,Home Delivery,Cash,Delivered,SKU00215,Matched,No,NaN,No Promotion,PUN-01,No
5,ORD1000005,2025-05-02,AHM-02,CUST005962,SKU01171,2,121.66,19.08,NaN,Store Pickup,UPI,Delivered,SKU01171,Matched,No,NaN,No Promotion,AHM-02,No
6,ORD1000006,2025-05-26,CHE-01,CUST010497,SKU00267,3,179.08,50.64,NaN,Store Pickup,Wallet,Delivered,SKU00267,Matched,No,NaN,No Promotion,CHE-01,No
7,ORD1000007,2025-06-01,CHE-01,CUST004609,SKU00418,4,263.93,279.84,PR0065,Store Pickup,UPI,Delivered,SKU00418,Matched,No,PR0065,Matched,CHE-01,No
8,ORD1000008,2025-04-13,CHE-03,CUST005707,SKU00540,2,62.93,3.86,NaN,Home Delivery,UPI,Delivered,SKU00540,Matched,No,NaN,No Promotion,CHE-03,No
9,ORD1000009,2025-06-16,BEN-03,CUST001059,SKU00396,2,149.88,112.80,PR0127,Home Delivery,NetBanking,Delivered,SKU00396,Matched,No,PR0127,Matched,BEN-03,No


In [360]:
# Calculate SKU-level profitability for promotion candidates

promo_candidates = sales_clean[
    (sales_clean["Order_Status"] == "Delivered") &
    (sales_clean["Negative_Quantity_Flag"] != "Yes") &
    (sales_clean["Quantity"] > 0) &
    (sales_clean["Selling_Price"].notna()) &
    (sales_clean["Discount_Amount"].notna()) & (sales_clean["Promo_ID"].isna())
].copy()

# Calculate net sales after discount
promo_candidates["Net_Sales"] = (promo_candidates["Quantity"] * promo_candidates["Selling_Price"]- promo_candidates["Discount_Amount"])
promo_candidates.head()

,Order_ID,Order_Date,Store_ID,Customer_ID,SKU_ID,Quantity,Selling_Price,Discount_Amount,Promo_ID,Delivery_Type,Payment_Mode,Order_Status,Original_SKU_ID,SKU_Mapping_Status,Negative_Quantity_Flag,Original_Promo_ID,Promo_Mapping_Status,Original_Store_ID,Cancellation_Flag,Net_Sales
0,ORD1000000,2025-06-01,HYD-03,CUST007684,SKU00565,4,72.73,19.52,<NA>,Home Delivery,UPI,Delivered,SKU00565,Matched,No,PROMO-9999,Missing,HYD-03,No,271.40
1,ORD1000001,2025-03-14,HYD-03,CUST004865,SKU01082,6,273.62,173.52,NaN,Store Pickup,Cash,Delivered,SKU01082,Matched,No,NaN,No Promotion,HYD-03,No,1468.20
2,ORD1000002,2025-06-05,BEN-05,CUST006245,SKU00901,2,58.95,4.42,NaN,Store Pickup,Card,Delivered,SKU00901,Matched,No,NaN,No Promotion,BEN-05,No,113.48
4,ORD1000004,2025-03-01,PUN-01,CUST000981,SKU00215,2,91.13,16.68,NaN,Home Delivery,Cash,Delivered,SKU00215,Matched,No,NaN,No Promotion,PUN-01,No,165.58
5,ORD1000005,2025-05-02,AHM-02,CUST005962,SKU01171,2,121.66,19.08,NaN,Store Pickup,UPI,Delivered,SKU01171,Matched,No,NaN,No Promotion,AHM-02,No,224.24


In [361]:
# Calculate SKU-level sales metrics
promo_sku = (promo_candidates.groupby("SKU_ID")
    .agg(Units_Sold=("Quantity", "sum"),Net_Sales=("Net_Sales", "sum"),
         Total_Discount=("Discount_Amount", "sum"),Order_Count=("Order_ID", "nunique")).reset_index())

# Add product cost. Until now, our summary contains SKU IDs and sales measures. 
# We need product names, categories, and unit costs to interpret profitability.
promo_sku = promo_sku.merge(product_clean[["SKU_ID", "SKU_Name", "Category", "Unit_Cost"]],on="SKU_ID",how="inner")

promo_sku.head()

,SKU_ID,Units_Sold,Net_Sales,Total_Discount,Order_Count,SKU_Name,Category,Unit_Cost
0,SKU00001,132,9808.08,1629.96,47,Snackify Biscuits 500g,Snacks,55.38
1,SKU00002,85,3941.52,539.29,32,FreshCo Paneer 250g,Dairy,43.66
2,SKU00003,78,6464.32,893.26,30,DailyDairy Edible Oil 500g,Groceries,76.77
3,SKU00004,89,17261.63,2909.37,31,Snackify Butter 1kg,Dairy,144.24
4,SKU00005,116,15830.58,2281.69,44,BabyBloom Detergent 750ml,Home Care,112.73


In [362]:
# Calculate gross profit after discounts
promo_sku["Net_Profit"] = (promo_sku["Net_Sales"]- (promo_sku["Units_Sold"] * promo_sku["Unit_Cost"]))
promo_sku.head()

,SKU_ID,Units_Sold,Net_Sales,Total_Discount,Order_Count,SKU_Name,Category,Unit_Cost,Net_Profit
0,SKU00001,132,9808.08,1629.96,47,Snackify Biscuits 500g,Snacks,55.38,2497.92
1,SKU00002,85,3941.52,539.29,32,FreshCo Paneer 250g,Dairy,43.66,230.42
2,SKU00003,78,6464.32,893.26,30,DailyDairy Edible Oil 500g,Groceries,76.77,476.26
3,SKU00004,89,17261.63,2909.37,31,Snackify Butter 1kg,Dairy,144.24,4424.27
4,SKU00005,116,15830.58,2281.69,44,BabyBloom Detergent 750ml,Home Care,112.73,2753.90


In [363]:
# Rank by net profit
best_5_skus = (promo_sku[promo_sku["Unit_Cost"].notna() &(promo_sku["Unit_Cost"] > 0)].sort_values("Net_Profit", ascending=False).head(5))

best_5_skus

,SKU_ID,Units_Sold,Net_Sales,Total_Discount,Order_Count,SKU_Name,Category,Unit_Cost,Net_Profit
878,SKU00885,132,56020.16,7346.12,48,CleanPro Powder 2L,Baby Care,301.31,16247.24
1129,SKU01136,166,57468.32,9513.90,52,DailyDairy Curd 500g,Dairy,265.46,13401.96
807,SKU00814,130,45785.20,5411.65,43,FrostBite Detergent 750ml,Home Care,251.61,13075.90
405,SKU00409,162,53991.62,9929.66,51,FrostBite Spices 1kg,Groceries,254.18,12814.46
858,SKU00865,115,71895.31,10464.62,40,Nestora Tissues 1L,Home Care,516.91,12450.66


In [365]:
# before exporting insights validate all required tasks for report
print("Top 5 Loss Drivers:", len(top_5_loss_drivers)); 
print("Worst 3 Vendors:", len(worst_3_vendors)); 
print("Stockout Hotspots:", len(stockout_hotspots)); 
print("Best 5 SKUs to Promote:", len(best_5_skus))
print(segment_summary); 
print(customer_metrics.columns.tolist())

Top 5 Loss Drivers: 5
Worst 3 Vendors: 3
Stockout Hotspots: 11
Best 5 SKUs to Promote: 5
                   Customer_Segment  Customer_Count  Customer_Percentage
0             High-Ticket Customers            1732                11.62
1  High-Value High-Return Customers            3082                20.68
2        High-Value Loyal Customers            3759                25.23
3          Low-Engagement Customers            6328                42.47
['Customer_ID', 'Total_Spend', 'Average_Order_Value', 'Order_Frequency', 'Return_Frequency', 'Cluster', 'Customer_Segment']



## Automated Insights Report – Excel and Executive PDF

This section consolidates the key business findings identified during the Python analysis.

The Excel report contains:
1. Top 5 Profit Leakage Drivers
2. Worst 3 Vendors by SLA Breach
3. Stockout Hotspots
4. Best 5 SKUs to Promote
5. Customer Segmentation Summary

# Create the Executive PDF
## Automated Executive Insights Report – PDF

This section creates a concise 1–2 page executive report summarizing the major profit leakage drivers, vendor performance issues, stockout impact, and potential promotion candidates identified during the analysis.

In [372]:
top_5_loss_drivers = top_5_loss_drivers.round(2); 
worst_3_vendors = worst_3_vendors.round(2); 
stockout_hotspots = stockout_hotspots.round(2); 
best_5_skus = best_5_skus.round(2); 
segment_summary = segment_summary.round(2)

In [373]:
%pip install reportlab
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4, landscape

styles = getSampleStyleSheet()
doc = SimpleDocTemplate("Retail_Executive_Summary.pdf", pagesize=landscape(A4), rightMargin=25, leftMargin=25, topMargin=25, bottomMargin=25)
story = []

story.append(Paragraph("Retail Chain Profit Leakage & Demand Optimization", styles["Title"]))
story.append(Paragraph("Executive Summary", styles["Heading1"]))
story.append(Spacer(1, 10))

def add_section(title, df):
    story.append(Paragraph(title, styles["Heading2"]))
    data = [list(df.columns)] + df.astype(str).values.tolist()
    table = Table(data, repeatRows=1, hAlign="LEFT")
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.darkblue),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("FONTSIZE", (0, 0), (-1, -1), 7),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.lightgrey])
    ]))
    story.append(table)
    story.append(Spacer(1, 10))

add_section("1. Top 5 Profit Leakage Drivers", top_5_loss_drivers)
add_section("2. Worst 3 Vendors", worst_3_vendors)
add_section("3. Stockout Hotspots", stockout_hotspots)
add_section("4. Best 5 SKUs to Promote", best_5_skus)
add_section("5. Customer Segmentation", segment_summary)

doc.build(story)
print("Executive PDF created successfully.")

Note: you may need to restart the kernel to use updated packages.
Executive PDF created successfully.


### Insights EXCEL report

In [374]:
with pd.ExcelWriter("Retail_Insights_Report.xlsx", engine="openpyxl") as writer:
    top_5_loss_drivers.to_excel(writer, sheet_name="Top_5_Loss_Drivers", index=False)
    worst_3_vendors.to_excel(writer, sheet_name="Worst_3_Vendors", index=False)
    stockout_hotspots.to_excel(writer, sheet_name="Stockout_Hotspots", index=False)
    best_5_skus.to_excel(writer, sheet_name="Best_5_SKUs_Promote", index=False)
    segment_summary.to_excel(writer, sheet_name="Segment_Summary", index=False)


In [368]:
print(os.path.abspath("Retail_Insights_Report.xlsx"))

C:\Users\manit\Retail Chain Profit Leakage & Demand Optimization\Retail_Insights_Report.xlsx


# Standalone Data Validation Script

This script validates the cleaned retail dataset without modifying the
cleaned data.

It checks missing values, duplicate records, numeric outliers, data
quality flags, and key validation results identified during the project.

The validation results are exported as a separate Excel Data Quality Report.

In [379]:

%%writefile data_validation.py

"""
Retail Chain Profit Leakage & Demand Optimization
Script: data_validation.py

Purpose:
1. Read all seven tables from the raw Excel workbook.
2. Calculate missing-value counts and percentages.
3. Count exact duplicate rows.
4. Identify numeric outliers using the IQR method.
5. Export the findings to an Excel Data Quality Report.
"""

import pandas as pd

# 1. File configuration
INPUT_FILE = r"C:\Users\manit\OneDrive\Desktop\PROJECT_1\Retail_Profit_Leakage_Dataset.xlsx"
OUTPUT_FILE = "Retail_Data_Quality_Report.xlsx"

# 2. Read all seven source tables
excel_data = pd.ExcelFile(INPUT_FILE)

table_names = [
    "Sales_Transactions",
    "Product_Master",
    "Inventory_Daily",
    "Returns",
    "Vendor_Supply",
    "Promotions",
    "Customer_Master"
]

null_results = []
duplicate_results = []
outlier_results = []

# 3. Check each table
for table_name in table_names:
    df = pd.read_excel(excel_data, sheet_name=table_name)
    total_rows = len(df)
    # Missing-value analysis
    for column in df.columns:
        null_count = df[column].isna().sum()
        null_results.append({
            "Table": table_name,
            "Column": column,
            "Null_Count": int(null_count),
            "Null_Percentage": round(null_count / total_rows * 100, 2) if total_rows > 0 else 0,
            "Total_Rows": total_rows})

    # Exact duplicate analysis
    duplicate_count = int(df.duplicated().sum())
    duplicate_results.append({"Table": table_name,"Total_Rows": total_rows,"Exact_Duplicate_Rows": duplicate_count})
    # Numeric outlier analysis using IQR
    numeric_columns = df.select_dtypes(include="number").columns
    for column in numeric_columns:
        values = df[column].dropna()
        if len(values) == 0:
            continue
        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1
        lower_limit = q1 - 1.5 * iqr
        upper_limit = q3 + 1.5 * iqr
        outlier_count = int(((values < lower_limit) |(values > upper_limit)).sum())
        outlier_results.append({
            "Table": table_name,
            "Column": column,
            "Outlier_Count": outlier_count,
            "Lower_Limit": round(lower_limit, 2),
            "Upper_Limit": round(upper_limit, 2)})

# 4. Convert the results into DataFrames
null_report = pd.DataFrame(null_results)
duplicate_report = pd.DataFrame(duplicate_results)
outlier_report = pd.DataFrame(outlier_results)

# 5. Export the Data Quality Report
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    null_report.to_excel(writer, sheet_name="Null_Report", index=False)
    duplicate_report.to_excel(writer, sheet_name="Duplicate_Report", index=False)
    outlier_report.to_excel(writer, sheet_name="Outlier_Report", index=False)

print("Data validation completed successfully.")
print("\nDuplicate summary:")
print(duplicate_report.to_string(index=False))
print("\nReport saved to:", OUTPUT_FILE)

Overwriting data_validation.py


In [380]:
!python data_validation.py

Data validation completed successfully.

Duplicate summary:
             Table  Total_Rows  Exact_Duplicate_Rows
Sales_Transactions       81600                  1217
    Product_Master        1218                    16
   Inventory_Daily       50500                   424
           Returns        4701                    84
     Vendor_Supply       30450                   365
        Promotions         210                     8
   Customer_Master       15300                   264

Report saved to: Retail_Data_Quality_Report.xlsx


# Standalone Customer Segmentation Script

This script performs customer segmentation using spending, order
frequency, average order value, and return frequency.

K-Means clustering is used to group customers into four segments.
The results are exported to a separate Excel workbook.

In [375]:

%%writefile customer_segmentation.py

"""
Retail Chain Profit Leakage & Demand Optimization
Script: customer_segmentation.py

Purpose:
1. Read cleaned Sales and Returns data.
2. Calculate customer spending and purchase behaviour.
3. Apply K-Means clustering to create four customer segments.
4. Export customer-level results and a segment summary.
"""

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. File configuration
INPUT_FILE = "Retail_Cleaned_Data.xlsx"
OUTPUT_FILE = "Customer_Segmentation_Report.xlsx"

# 2. Read cleaned data
sales_clean = pd.read_excel(INPUT_FILE, sheet_name="Sales")
returns_clean = pd.read_excel(INPUT_FILE, sheet_name="Returns")

# 3. Select valid sales records
valid_sales = sales_clean[
    (sales_clean["Order_Status"] == "Delivered") &
    (sales_clean["Negative_Quantity_Flag"] != "Yes") &
    (sales_clean["Quantity"] > 0) &
    (sales_clean["Customer_ID"].notna()) &
    (sales_clean["Selling_Price"].notna()) &
    (sales_clean["Discount_Amount"].notna())
].copy()

# 4. Calculate net sales after discounts
valid_sales["Net_Sales"] = (valid_sales["Quantity"] * valid_sales["Selling_Price"]- valid_sales["Discount_Amount"])

# 5. Calculate customer-level metrics
customer_metrics = valid_sales.groupby("Customer_ID").agg(Total_Spend=("Net_Sales", "sum"),Order_Frequency=("Order_ID", "nunique")).reset_index()

customer_metrics["Average_Order_Value"] = (customer_metrics["Total_Spend"] /customer_metrics["Order_Frequency"])

# 6. Link returns to customers through Order_ID
return_customer = returns_clean.merge(sales_clean[["Order_ID", "Customer_ID"]],on="Order_ID",how="left")

return_frequency = (return_customer.dropna(subset=["Customer_ID"]).
    groupby("Customer_ID").agg(Return_Frequency=("Return_ID", "nunique")).reset_index())

# 7. Add return frequency to customer metrics
customer_metrics = customer_metrics.merge(return_frequency,on="Customer_ID",how="left")

customer_metrics["Return_Frequency"] = (customer_metrics["Return_Frequency"].fillna(0).astype(int))

# 8. Prepare the four clustering features
segmentation_features = [
    "Total_Spend",
    "Average_Order_Value",
    "Order_Frequency",
    "Return_Frequency"
]

# 9. Standardize the features
scaler = StandardScaler()
customer_scaled = scaler.fit_transform(customer_metrics[segmentation_features])

# 10. Create four customer clusters
kmeans_final = KMeans(n_clusters=4,random_state=42,n_init=10)
customer_metrics["Cluster"] = kmeans_final.fit_predict(customer_scaled)

# 11. Assign the segment names from the completed analysis
segment_names = {
    0: "High-Value Loyal Customers",
    1: "Low-Engagement Customers",
    2: "High-Value High-Return Customers",
    3: "High-Ticket Customers"}

customer_metrics["Customer_Segment"] = (customer_metrics["Cluster"].map(segment_names))

# 12. Summarize the customer segments
segment_summary = customer_metrics.groupby("Customer_Segment").agg(Customer_Count=("Customer_ID", "nunique")).reset_index()
segment_summary["Customer_Percentage"] = (segment_summary["Customer_Count"] /segment_summary["Customer_Count"].sum() * 100).round(2)

# 13. Export the results
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    customer_metrics.to_excel(writer, sheet_name="Customer_Details", index=False)
    segment_summary.to_excel(writer, sheet_name="Segment_Summary", index=False)

print("Customer segmentation completed successfully.")
print("Customers segmented:", len(customer_metrics))
print("\nSegment summary:")
print(segment_summary.to_string(index=False))
print("\nReport saved to:", OUTPUT_FILE)

Overwriting customer_segmentation.py


In [376]:
!python customer_segmentation.py

Customer segmentation completed successfully.
Customers segmented: 14901

Segment summary:
                Customer_Segment  Customer_Count  Customer_Percentage
           High-Ticket Customers            1732                11.62
High-Value High-Return Customers            3082                20.68
      High-Value Loyal Customers            3759                25.23
        Low-Engagement Customers            6328                42.47

Report saved to: Customer_Segmentation_Report.xlsx


# Automated Insights Report 
## using .py scripts

This section creates a standalone Python script to automatically generate
key business insights from the cleaned retail dataset.

The script generates the following outputs:

- Top 5 loss-making SKUs
- Worst 3 vendors based on SLA breach
- Validated stockout hotspots
- Best 5 SKUs to promote
- Customer segmentation summary

The results are exported in two formats:

1. Excel summary report
2. PDF executive report

The script reads the cleaned data and does not modify the original
cleaned workbook.

In [391]:

%%writefile automated_insights_report.py

"""
Retail Chain Profit Leakage & Demand Optimization
Automated Insights Report

Generates:
1. Top 5 Profit Leakage Drivers
2. Worst 3 Vendors
3. Stockout Hotspots
4. Best 5 SKUs to Promote
5. Customer Segment Summary

Exports:
- Retail_Insights_Report.xlsx
- Retail_Executive_Summary.pdf
"""

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4, landscape

# =========================================================
# 1. FILE CONFIGURATION
# =========================================================

INPUT_FILE = r"C:\Users\manit\Retail Chain Profit Leakage & Demand Optimization\Retail_Cleaned_Data.xlsx"
EXCEL_OUTPUT = "Retail_Insights_Report_automated.xlsx"
PDF_OUTPUT = "Retail_Executive_Summary_automated.pdf"

# =========================================================
# 2. READ CLEANED DATA
# =========================================================
sales_clean = pd.read_excel(INPUT_FILE, sheet_name="Sales")
product_clean = pd.read_excel(INPUT_FILE, sheet_name="Product_Master")
inventory_clean = pd.read_excel(INPUT_FILE, sheet_name="Inventory_Daily")
returns_clean = pd.read_excel(INPUT_FILE, sheet_name="Returns")
vendor_clean = pd.read_excel(INPUT_FILE, sheet_name="Vendor_Supply")

# Ensure date columns are datetime values
sales_clean["Order_Date"] = pd.to_datetime(sales_clean["Order_Date"], errors="coerce")
inventory_clean["Date"] = pd.to_datetime(inventory_clean["Date"], errors="coerce")
vendor_clean["PO_Date"] = pd.to_datetime(vendor_clean["PO_Date"], errors="coerce")
vendor_clean["Expected_Delivery_Date"] = pd.to_datetime(vendor_clean["Expected_Delivery_Date"], errors="coerce")
vendor_clean["Actual_Delivery_Date"] = pd.to_datetime(vendor_clean["Actual_Delivery_Date"], errors="coerce")

# =========================================================
# 3. TOP 5 PROFIT LEAKAGE DRIVERS
# =========================================================
profit_sales = sales_clean[
    (sales_clean["Order_Status"] == "Delivered") &
    (sales_clean["Negative_Quantity_Flag"] != "Yes") &
    (sales_clean["Quantity"] > 0) &
    sales_clean["Selling_Price"].notna() &
    sales_clean["Discount_Amount"].notna()
].copy()

profit_sales["Net_Sales"] = (profit_sales["Quantity"] * profit_sales["Selling_Price"]- profit_sales["Discount_Amount"])
profit_sales = profit_sales.merge(product_clean[["SKU_ID", "SKU_Name", "Category", "Unit_Cost"]],on="SKU_ID",how="inner")
profit_sales = profit_sales[profit_sales["Unit_Cost"].notna() &(profit_sales["Unit_Cost"] > 0)].copy()
profit_sales["Gross_Profit"] = (profit_sales["Net_Sales"]  - profit_sales["Quantity"] * profit_sales["Unit_Cost"])
approved_returns = (returns_clean[returns_clean["Return_Status"]=="Approved"].groupby("SKU_ID")["Refund_Amount"].sum().reset_index(name="Total_Refund"))

profit_leakage = (profit_sales.groupby(["SKU_ID", "SKU_Name", "Category"], as_index=False).agg(Gross_Profit=("Gross_Profit", "sum"))
    .merge(approved_returns, on="SKU_ID", how="left"))

profit_leakage["Total_Refund"] = profit_leakage["Total_Refund"].fillna(0)
profit_leakage["Net_Profit"] = (profit_leakage["Gross_Profit"] - profit_leakage["Total_Refund"])

top_5_loss_drivers = (profit_leakage.sort_values("Net_Profit").head(5).round(2))

# =========================================================
# 4. WORST 3 VENDORS
# =========================================================
valid_vendor = vendor_clean[
    vendor_clean["PO_Date"].notna() &
    vendor_clean["Expected_Delivery_Date"].notna() &
    vendor_clean["Actual_Delivery_Date"].notna() &
    vendor_clean["Vendor_ID"].notna() &
    vendor_clean["PO_ID"].notna() &
    (vendor_clean["Data_Quality_Flag"].fillna("Normal") != "ImpossibleDateCombo")
].copy()

valid_vendor["Late"] = (valid_vendor["Actual_Delivery_Date"] >valid_vendor["Expected_Delivery_Date"])

vendor_performance = (valid_vendor.groupby("Vendor_ID").agg(
        Total_PO_Count=("PO_ID", "count"),
        Late_PO_Count=("Late", "sum"),
        Average_Delay_Days=("Delay_Days", "mean")).reset_index())

vendor_performance["SLA_Breach_Percent"] = (vendor_performance["Late_PO_Count"] /vendor_performance["Total_PO_Count"] * 100).round(2)

worst_3_vendors = (vendor_performance.sort_values(["SLA_Breach_Percent", "Average_Delay_Days"],ascending=[False, False]).head(3).round(2))

# =========================================================
# 5. STOCKOUT HOTSPOTS
# Store_ID is preserved in its original CHE-02 format.
# =========================================================

stockout_inventory = inventory_clean[
    (inventory_clean["Closing_Stock"] == 0) &
    inventory_clean["SKU_ID"].notna() &
    inventory_clean["Store_ID"].notna() &
    inventory_clean["Date"].notna()
].copy()

stockout_sales = sales_clean[
    (sales_clean["Order_Status"] == "Delivered") &
    (sales_clean["Negative_Quantity_Flag"] != "Yes") &
    (sales_clean["Quantity"] > 0) &
    sales_clean["Selling_Price"].notna() &
    sales_clean["Order_Date"].notna()
].copy()

stockout_matches = stockout_inventory.merge(stockout_sales,
    left_on=["SKU_ID", "Store_ID", "Date"],
    right_on=["SKU_ID", "Store_ID", "Order_Date"],
    how="inner",suffixes=("_Inventory", "_Sales"))

stockout_matches["Potential_Sales_Value"] = (stockout_matches["Quantity"] *stockout_matches["Selling_Price"])
stockout_hotspots = (stockout_matches.groupby(["SKU_ID", "Store_ID"],as_index=False).agg(
        Stockout_Days=("Date", "nunique"),
        Demand_Quantity=("Quantity", "sum"),
        Potential_Sales_Value=("Potential_Sales_Value", "sum")).sort_values("Potential_Sales_Value", ascending=False).round(2))

# =========================================================
# 6. BEST 5 SKUs TO PROMOTE
# =========================================================
promo_candidates = sales_clean[
    (sales_clean["Order_Status"] == "Delivered") &
    (sales_clean["Negative_Quantity_Flag"] != "Yes") &
    (sales_clean["Quantity"] > 0) &
    sales_clean["Selling_Price"].notna() &
    sales_clean["Discount_Amount"].notna() &
    sales_clean["Promo_ID"].isna()
].copy()

promo_candidates["Net_Sales"] = (promo_candidates["Quantity"] * promo_candidates["Selling_Price"]- promo_candidates["Discount_Amount"])
promo_sku = (promo_candidates.groupby("SKU_ID").agg(
        Units_Sold=("Quantity", "sum"),
        Net_Sales=("Net_Sales", "sum"),
        Total_Discount=("Discount_Amount", "sum"),
        Order_Count=("Order_ID", "nunique")).reset_index().merge(product_clean[["SKU_ID", "SKU_Name", "Category", "Unit_Cost"]],on="SKU_ID",how="inner"))

promo_sku["Net_Profit"] = (
promo_sku["Net_Sales"]- promo_sku["Units_Sold"] * promo_sku["Unit_Cost"])
best_5_skus = (promo_sku[promo_sku["Unit_Cost"].notna() &(promo_sku["Unit_Cost"] > 0)].sort_values("Net_Profit", ascending=False).head(5).round(2))

# =========================================================
# 7. CUSTOMER SEGMENT SUMMARY
# =========================================================
customer_sales = sales_clean[
    (sales_clean["Order_Status"] == "Delivered") &
    (sales_clean["Negative_Quantity_Flag"] != "Yes") &
    (sales_clean["Quantity"] > 0) &
    sales_clean["Customer_ID"].notna() &
    sales_clean["Selling_Price"].notna() &
    sales_clean["Discount_Amount"].notna()
].copy()

customer_sales["Net_Sales"] = (customer_sales["Quantity"] * customer_sales["Selling_Price"]- customer_sales["Discount_Amount"])
customer_metrics = customer_sales.groupby("Customer_ID").agg(Total_Spend=("Net_Sales", "sum"),Order_Frequency=("Order_ID", "nunique")).reset_index()
customer_metrics["Average_Order_Value"] = (customer_metrics["Total_Spend"] /customer_metrics["Order_Frequency"])
return_customer = returns_clean.merge(sales_clean[["Order_ID", "Customer_ID"]], on="Order_ID",how="left")

return_frequency = (return_customer.dropna(subset=["Customer_ID"]).groupby("Customer_ID").agg(Return_Frequency=("Return_ID", "nunique")).reset_index())
customer_metrics = customer_metrics.merge(return_frequency,on="Customer_ID",how="left")
customer_metrics["Return_Frequency"] = (customer_metrics["Return_Frequency"].fillna(0).astype(int))

features = [
    "Total_Spend",
    "Average_Order_Value",
    "Order_Frequency",
    "Return_Frequency"
]

scaler = StandardScaler()
customer_scaled = scaler.fit_transform(customer_metrics[features])
model = KMeans(n_clusters=4, random_state=42, n_init=10)
customer_metrics["Cluster"] = model.fit_predict(customer_scaled)

# Assign names using cluster profiles
profiles = customer_metrics.groupby("Cluster").agg(
    Spend=("Total_Spend", "mean"),
    AOV=("Average_Order_Value", "mean"),
    Orders=("Order_Frequency", "mean"),
    Returns=("Return_Frequency", "mean"))

loyal_cluster = profiles.sort_values(["Orders", "Returns"], ascending=[False, True]).index[0]
high_return_cluster = profiles.drop(index=loyal_cluster).sort_values("Returns", ascending=False).index[0]
remaining = profiles.drop(index=[loyal_cluster, high_return_cluster])
high_ticket_cluster = remaining.sort_values("AOV", ascending=False).index[0]
low_engagement_cluster = remaining.index.difference([high_ticket_cluster])[0]

segment_names = {
    loyal_cluster: "High-Value Loyal Customers",
    low_engagement_cluster: "Low-Engagement Customers",
    high_return_cluster: "High-Value High-Return Customers",
    high_ticket_cluster: "High-Ticket Customers"
}

customer_metrics["Customer_Segment"] = (customer_metrics["Cluster"].map(segment_names))
segment_summary = (customer_metrics.groupby("Customer_Segment").agg(Customer_Count=("Customer_ID", "nunique")).reset_index())
segment_summary["Customer_Percentage"] = (segment_summary["Customer_Count"] /segment_summary["Customer_Count"].sum() * 100).round(2)

# =========================================================
# 8. EXPORT EXCEL SUMMARY
# =========================================================
with pd.ExcelWriter(EXCEL_OUTPUT, engine="openpyxl") as writer:
    top_5_loss_drivers.to_excel(writer, sheet_name="Top_5_Loss_Drivers", index=False)
    worst_3_vendors.to_excel(writer, sheet_name="Worst_3_Vendors", index=False)
    stockout_hotspots.to_excel(writer, sheet_name="Stockout_Hotspots", index=False)
    best_5_skus.to_excel(writer, sheet_name="Best_5_SKUs_Promote", index=False)
    segment_summary.to_excel(writer, sheet_name="Segment_Summary", index=False)

# =========================================================
# 9. GENERATE PDF REPORT
# =========================================================
styles = getSampleStyleSheet()
doc = SimpleDocTemplate(
    PDF_OUTPUT,
    pagesize=landscape(A4),
    rightMargin=25,
    leftMargin=25,
    topMargin=25,
    bottomMargin=25)

story = [Paragraph("Retail Chain Profit Leakage &amp; Demand Optimization",styles["Title"]),
    Paragraph("Automated Executive Insights Report", styles["Heading1"]),
    Spacer(1, 10)]

def add_pdf_section(title, df):
    story.append(Paragraph(title, styles["Heading2"]))
    data = [list(df.columns)] + df.astype(str).values.tolist()
    table = Table(data, repeatRows=1, hAlign="LEFT")
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.darkblue),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("FONTSIZE", (0, 0), (-1, -1), 7),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.lightgrey])]))
    story.append(table)
    story.append(Spacer(1, 10))

add_pdf_section("1. Top 5 Profit Leakage Drivers", top_5_loss_drivers)
add_pdf_section("2. Worst 3 Vendors", worst_3_vendors)
add_pdf_section("3. Stockout Hotspots", stockout_hotspots)
add_pdf_section("4. Best 5 SKUs to Promote", best_5_skus)
add_pdf_section("5. Customer Segmentation", segment_summary)

doc.build(story)

# =========================================================
# 10. COMPLETION MESSAGE
# =========================================================
print("Automated insights report generated successfully.")
print("Top 5 loss drivers:", len(top_5_loss_drivers))
print("Worst 3 vendors:", len(worst_3_vendors))
print("Stockout hotspot rows:", len(stockout_hotspots))
print("Best 5 SKUs:", len(best_5_skus))
print("Customer segments:", len(segment_summary))
print("Excel report:", EXCEL_OUTPUT)
print("PDF report:", PDF_OUTPUT)

Overwriting automated_insights_report.py


In [392]:
!python automated_insights_report.py

Automated insights report generated successfully.
Top 5 loss drivers: 5
Worst 3 vendors: 3
Stockout hotspot rows: 10
Best 5 SKUs: 5
Customer segments: 4
Excel report: Retail_Insights_Report_automated.xlsx
PDF report: Retail_Executive_Summary_automated.pdf
